In [ ]:
# =============================================================================
# CELL 1 - COLAB ENVIRONMENT SETUP
# =============================================================================
# Installs, GPU check, Google Drive mount and Hugging Face login.
# facebook/sam3 is a GATED model: you must accept the license at
# https://huggingface.co/facebook/sam3 with the same account as your token.
# facebook/dinov2-large (exemplar embeddings) is NOT gated - no extra step.
# =============================================================================

import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

!pip install -q -U transformers accelerate huggingface_hub supervision opencv-python-headless

import torch
import torchvision

print("PyTorch version    :", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA available     :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU                :", torch.cuda.get_device_name(0))
    print("GPU memory (GB)    :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

from google.colab import drive
drive.mount('/content/drive')

from huggingface_hub import login
login()  # uses HF_TOKEN from Colab secrets, or paste your token (needs access to facebook/sam3)

print("Environment ready.")


In [ ]:
# =============================================================================
# CELL 2 - IMPORTS
# =============================================================================

import os
import gc
import csv
import time
import itertools
import hashlib

import numpy as np
import pandas as pd
import torch

from PIL import Image, ImageFilter
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.lines import Line2D

import supervision as sv
from supervision.metrics import MeanAveragePrecision

from transformers import Sam3Model, Sam3Processor
from transformers import AutoImageProcessor, AutoModel   # DINOv2 exemplar embeddings

# UAV orthomosaics are very large; disable PIL's decompression-bomb guard.
Image.MAX_IMAGE_PIXELS = None

print("Imports OK. supervision version:", sv.__version__)


In [ ]:
# =============================================================================
# CELL 3 - CONFIGURATION
# =============================================================================
# Every parameter of the study lives here so that a thesis experiment is fully
# described by this single cell.
#
# PIPELINE = the diversity pipeline of k_diverse_exemplars_no_tiling, but SAM3
# now sees OVERLAPPING TILES instead of the downscaled whole image.
#
#   ONCE PER IMAGE (full resolution, before any tiling):
#     crop every GT Rumex   GT bbox -> +10% context -> square padding -> 224x224
#     -> DINOv2 CLS embedding, L2-normalised
#     -> pairwise cosine distance  d(i,j) = 1 - cos(z_i, z_j)
#     -> diversity selection  S* = argmax_{|S|=3} min_{i,j in S} d(i,j)
#     -> 3 GT plants = the exemplar prompts of this image
#
#   THEN, FOR EVERY TILE of that image:
#     the tile is left untouched and the 3 exemplar crops are pasted into a STRIP
#     above it (textured local background, feathered edges). The exemplar boxes
#     INSIDE THAT STRIP are the SAM3 positive box prompts. The strip is needed
#     because the 3 selected plants are usually NOT inside the tile being scanned.
#     Detections that belong to the strip instead of the tile are removed, the
#     rest are converted to original-image coordinates.
#
#   Detections of all tiles are pooled -> offline NMS merges the duplicates that
#   the overlap produces -> evaluation is exactly the one of the no-tiling
#   notebook (all_gt / held_out, same matcher, same thresholds).
#
# The selection is deterministic, so there is exactly ONE run per image
# (no anchors, no random exemplar sampling).
# =============================================================================

# ------------------------- experiment identity -------------------------------
EXPERIMENT_NAME = "k_diverse_exemplars_tiling"   # DINOv2 diversity exemplars + tiling
K_EXEMPLARS     = 3           # D1 + D2 + D3
N_EXEMPLARS     = K_EXEMPLARS
USE_TILING      = True        # overlapping tiles (False -> whole image as one tile)
PROMPT_TYPE     = "diversity_dinov2_tiling"

# ------------------------------- dataset -------------------------------------
IMAGES_ROOT = "/content/drive/MyDrive/master_thesis/dataset/AGS_Multi_Rumex/images"
LABELS_ROOT = "/content/drive/MyDrive/master_thesis/dataset/AGS_Multi_Rumex/annotations_yolo"
RUMEX_CLASS_ID = 0
VALID_IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")

# ------------------------------- outputs -------------------------------------
RESULTS_ROOT = f"/content/drive/MyDrive/master_thesis/results_new1/{EXPERIMENT_NAME}"

# ------------------------------- tiling --------------------------------------
TILE_SIZE = 1000              # tile edge length in original-image pixels
OVERLAP   = 150               # overlap between neighbouring tiles in pixels
CACHE_TILES_IN_MEMORY = True  # crop all tiles once per image and keep them in RAM
# NOTE: there is no MAX_DIM any more. The whole-image downscale of the no-tiling
# experiment is exactly what tiling replaces: every tile goes to SAM3 at NATIVE
# resolution, which is the point of the comparison between the two notebooks.

# ------------------- DINOv2 embeddings of the GT crops ------------------------
# The crops are taken from the FULL-RESOLUTION image (never from a tile), so the
# selection is identical to the no-tiling experiment and completely independent
# of the tile grid.
#
#   GT bbox -> +10% fixed context -> square padding -> 224x224 -> DINOv2
#
DINOV2_MODEL_ID   = "facebook/dinov2-large"   # ViT-L/14, 1024-d CLS token
EMBEDDING_FEATURE = "cls"      # CLS token of the last hidden layer, then L2-normalised

CROP_MODE    = "context_pad_square"
CROP_CONTEXT = 0.10            # +10% of the box WIDTH left and right, +10% of the box
                               # HEIGHT top and bottom -> context box = 1.2 x 1.2 x GT box
CROP_PAD_FILL = (124, 116, 104)  # padding colour of the square canvas = ImageNet mean
                               # (neutral grey after DINOv2 normalisation). "edge"
                               # replicates the border pixels instead.

# The DINOv2 processor DEFAULTS to "resize shortest edge 256 + CENTER-CROP 224",
# which would cut the context margin away again -> resize directly to 224x224.
DINOV2_INPUT_SIZE   = 224
DINOV2_CENTER_CROP  = False

DINOV2_BATCH_SIZE = 16         # crops per forward pass
DINOV2_DTYPE      = torch.float32   # embeddings stay fp32 (SAM3 may run in fp16)

# The SAME padded square crops are pasted into the exemplar strip, so the plant
# SAM3 is shown is pixel-for-pixel the plant the selection compared.
EXEMPLAR_CROP_FOR_STRIP = "embedding_crop"   # "embedding_crop" | "tight_gt_box"

# --------------------- diversity-based exemplar selection ---------------------
# d(i,j) = 1 - cos(z_i, z_j)  on L2-normalised embeddings
# S* = argmax_{|S|=3} min_{i,j in S, i != j} d(i,j)
DIVERSITY_METRIC       = "cosine"     # distance = 1 - cosine similarity
DIVERSITY_OBJECTIVE    = "max_min"    # maximise the SMALLEST pairwise distance in the triple
DIVERSITY_EXACT_MAX_GT = 300          # up to this many GT boxes the search is EXHAUSTIVE

# --------------------------- SAM3 inference ----------------------------------
# The confidence threshold is FIXED from the start. It is used both
#   (a) inside SAM3 post-processing (every tile), and
#   (b) as the operating point for precision / recall / F1 / IoU1 / IoU2.
CONFIDENCE_THRESHOLD = 0.30
MASK_THRESHOLD = 0.40         # SAM3 mask binarisation (only used transiently)
BATCH_SIZE = 4                # composed tiles per forward pass (T4-friendly).
                              # Lower to 2 or 1 if you hit CUDA OOM.
USE_FP16 = True               # half precision inference on the GPU
KEEP_MASKS = False            # masks are NEVER stored; they are only used
                              # transiently for the mask-fill ratio.

# ------------------------- exemplar strip layout ------------------------------
STRIP_MARGIN = 6              # px between exemplar crops inside the strip
FEATHER_WIDTH = 8             # px of soft alpha blending around each exemplar crop
BACKGROUND_BLUR_RADIUS = 1.5  # light Gaussian blur applied to the sampled background
MAX_STRIP_HEIGHT_FRACTION = None   # optional clamp on the strip height (disabled)

# ------------------- plausibility filter (unchanged thresholds) ---------------
MIN_FILL_RATIO   = 0.15       # >=15 % of the box area must be covered by the mask
MAX_AREA_FRACTION = 0.80      # a detection may not cover >80 % of a tile
EDGE_MARGIN      = 5          # boxes with width or height <=5 px are discarded

# ---------------- target-region membership rule (CELL 13) ---------------------
# A prediction made on the composed image (strip on top + tile below) is kept only
# if at least this fraction of its AREA lies inside the real tile region.
TILE_REGION_MIN_FRACTION = 0.50

# ------------------------------ evaluation ------------------------------------
EVAL_IOU_THRESHOLD = 0.50     # IoU needed for a prediction to count as a TP
PROMPT_IGNORE_IOU  = 0.50     # held_out mode: an unmatched prediction whose best IoU
                              # with a PROMPT GT box is >= this value is IGNORED

# ------------------------------- NMS -----------------------------------------
# Fixed from the start, applied offline to the stored pre-NMS detections.
NMS_IOU_THRESHOLD = 0.40

EVALUATION_MODES = ["all_gt", "held_out"]
# all_gt   : every GT box of the image is evaluated (classical evaluation).
# held_out : the 3 prompt plants (D1, D2, D3) are IGNORED, and so are the
#            predictions that fall on them.

# --------------------------- qualitative plot ---------------------------------
PLOT_EVALUATION_MODE = "all_gt"   # per-image AP50 of this mode selects the image
PLOT_MIN_GT_BOXES    = 7          # the plotted image must have at least this many GT boxes
PLOT_MAX_DISPLAY_DIM = 2048       # display-only downscale (the full image is 8192 px wide)
PLOT_SHOW_SCORES     = True       # write the confidence next to each predicted box

print(f"Configuration loaded for experiment '{EXPERIMENT_NAME}'")
print(f"  results folder : {RESULTS_ROOT}")
print(f"  prompts        : {K_EXEMPLARS} GT boxes chosen by {DIVERSITY_OBJECTIVE} "
      f"{DIVERSITY_METRIC} diversity of {DINOV2_MODEL_ID} {EMBEDDING_FEATURE} embeddings")
print(f"  crops          : GT box +{CROP_CONTEXT:.0%} context -> square padding "
      f"-> {DINOV2_INPUT_SIZE}px (strip uses the {EXEMPLAR_CROP_FOR_STRIP})")
print(f"  tiling         : {USE_TILING} (TILE_SIZE={TILE_SIZE}, OVERLAP={OVERLAP})")
print(f"  confidence     : {CONFIDENCE_THRESHOLD} (fixed)")
print(f"  mask threshold : {MASK_THRESHOLD}")
print(f"  NMS IoU        : {NMS_IOU_THRESHOLD} (fixed)")
print(f"  batch / fp16   : {BATCH_SIZE} / {USE_FP16}")
print(f"  evaluation     : {EVALUATION_MODES}, IoU >= {EVAL_IOU_THRESHOLD}")
print(f"  masks stored   : {KEEP_MASKS}")


In [ ]:
# =============================================================================
# CELL 4 - OUTPUT FOLDERS
# =============================================================================
# results_new1/<EXPERIMENT_NAME>/
#   raw_detections/      pre-NMS detections (+ tile provenance and the GT
#                        embedding distance matrix), NPZ, one file per image
#   metrics/             per-image table, experiment summary, pooled dataset AP,
#                        size-group recall
#   confusion_matrices/  CSV + PNG for all_gt and held_out
#   plots/               qualitative GT-vs-prediction figure (CELL 30)
# =============================================================================

RAW_DETECTIONS_DIR    = os.path.join(RESULTS_ROOT, "raw_detections")
METRICS_DIR           = os.path.join(RESULTS_ROOT, "metrics")
CONFUSION_MATRIX_DIR  = os.path.join(RESULTS_ROOT, "confusion_matrices")
PLOTS_DIR             = os.path.join(RESULTS_ROOT, "plots")

for d in [RESULTS_ROOT, RAW_DETECTIONS_DIR, METRICS_DIR,
          CONFUSION_MATRIX_DIR, PLOTS_DIR]:
    os.makedirs(d, exist_ok=True)

# Manifest of finished inference runs -> used for crash-safe resuming.
# area_D1/D2/D3 are the areas of the three SELECTED plants (a RESULT of the
# diversity selection, not its criterion) and are logged so that the size profile
# of diversity prompts stays comparable with the S/M/L experiment.
RUN_MANIFEST_CSV = os.path.join(RAW_DETECTIONS_DIR, "runs_manifest.csv")
MANIFEST_COLUMNS = [
    "experiment_name", "image_ID", "Prompt_ID", "Prompt_Type",
    "n_gt", "n_prompt_gt",
    "area_D1_px2", "area_D2_px2", "area_D3_px2", "median_gt_area_px2",
    "min_pairwise_distance", "mean_pairwise_distance",
    "gt_mean_distance", "search_mode",
    "n_tiles", "n_detections_pre_nms", "image_width", "image_height",
    "npz_file", "embedding_seconds", "inference_seconds",
]

print("Output folders ready under:", RESULTS_ROOT)


In [ ]:
# =============================================================================
# CELL 5 - MODELS: SAM3 (detector, fp16) + DINOv2 (exemplar embeddings, fp32)
# =============================================================================
# With USE_FP16 the SAM3 weights are loaded in half precision, which roughly
# halves GPU memory and speeds up inference on a T4; inference additionally runs
# inside torch.inference_mode() + torch.autocast (see CELL 15). DINOv2 stays in
# fp32: it is only run once per image on a handful of small crops, and the cosine
# distances that drive the selection are compared, not trained.
# Both models stay resident on the GPU for the whole inference loop (CELL 17).
# =============================================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_DTYPE = torch.float16 if (USE_FP16 and device == "cuda") else torch.float32

sam3_model = Sam3Model.from_pretrained(
    "facebook/sam3",
    torch_dtype=MODEL_DTYPE,
    device_map="auto",
)
sam3_model.eval()

sam3_processor = Sam3Processor.from_pretrained("facebook/sam3")

print("SAM3 loaded.")
print("  model device:", next(sam3_model.parameters()).device)
print("  model dtype :", next(sam3_model.parameters()).dtype)

dino_processor = AutoImageProcessor.from_pretrained(DINOV2_MODEL_ID)
dino_model = AutoModel.from_pretrained(DINOV2_MODEL_ID, torch_dtype=DINOV2_DTYPE).to(device)
dino_model.eval()

DINOV2_EMBED_DIM = int(dino_model.config.hidden_size)

print(f"DINOv2 loaded ({DINOV2_MODEL_ID}).")
print("  model device:", next(dino_model.parameters()).device)
print("  model dtype :", next(dino_model.parameters()).dtype)
print("  embedding   :", EMBEDDING_FEATURE, f"({DINOV2_EMBED_DIM}-d)")


In [ ]:
# =============================================================================
# CELL 6 - STABLE REPRODUCIBILITY HELPER
# =============================================================================
# The exemplars themselves are chosen deterministically by embedding diversity
# (CELL 14), so the only random ingredient left in this notebook is the
# background patch sampled behind the exemplar crops in the strip (CELL 11).
#
# It is seeded with a SHA-256 digest of a plain text key, NOT with Python's
# hash(), which is randomised per process (PYTHONHASHSEED): the same image would
# otherwise get a different strip after a Colab restart. SHA-256 is a fixed
# mathematical function, so the same key always yields the same seed on every
# machine, every Python version and every session.
# =============================================================================

def stable_seed(*parts) -> int:
    """
    Deterministic 32-bit seed from any set of values.

    Input : any number of values (strings / ints) identifying the run,
            e.g. stable_seed(EXPERIMENT_NAME, image_id, "strip_bg")
    Output: int in [0, 2**32) - identical in every Python process, forever.
    """
    key = "|".join(str(p) for p in parts)
    digest = hashlib.sha256(key.encode("utf-8")).digest()
    return int.from_bytes(digest[:8], "big") % (2 ** 32)


_demo = stable_seed(EXPERIMENT_NAME, "folderA/img_001", "strip_bg")
print("stable_seed demo :", _demo, "(identical in every session / machine)")


In [ ]:
# =============================================================================
# CELL 7 - DATASET AND YOLO ANNOTATION HELPERS
# =============================================================================
# Unchanged logic from the original E02_1 notebook (it worked); only wrapped
# into functions and given comments, same as E02_2's CELL 8.
# =============================================================================

def load_yolo_boxes(label_path, img_width, img_height, class_id=RUMEX_CLASS_ID):
    """
    Read a YOLO .txt annotation file and convert it to pixel corner boxes.

    Input : label_path            - YOLO txt file
            img_width, img_height - size of the ORIGINAL image in pixels
            class_id              - keep only this class (0 = Rumex)
    Output: np.ndarray (N, 4) float32, boxes as [x1, y1, x2, y2] in pixels.

    YOLO stores normalised (class, x_center, y_center, width, height).
    """
    boxes = []
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue                      # skip empty lines
            if int(parts[0]) != class_id:
                continue                      # keep only the requested class
            xc, yc, bw, bh = map(float, parts[1:5])
            xc, yc = xc * img_width, yc * img_height
            bw, bh = bw * img_width, bh * img_height
            boxes.append([xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2])
    return np.array(boxes, dtype=np.float32).reshape(-1, 4)


def find_label_path(image_filename_no_ext, folder_name):
    """
    Locate the YOLO txt belonging to an image, supporting both a mirrored folder
    structure (labels/<folder>/<n>.txt) and a flat one (labels/<n>.txt).
    Returns the path or None.
    """
    mirrored = os.path.join(LABELS_ROOT, folder_name, image_filename_no_ext + ".txt")
    flat = os.path.join(LABELS_ROOT, image_filename_no_ext + ".txt")
    if os.path.exists(mirrored):
        return mirrored
    if os.path.exists(flat):
        return flat
    return None


def discover_images():
    """
    Walk IMAGES_ROOT and collect every image together with its label file.
    Output: list of (folder, image_path, label_path, image_id) tuples,
            where image_id = "<folder>/<image name without extension>".
    """
    records = []
    for folder in sorted(os.listdir(IMAGES_ROOT)):
        folder_path = os.path.join(IMAGES_ROOT, folder)
        if not os.path.isdir(folder_path):
            continue
        for fname in sorted(os.listdir(folder_path)):
            if not fname.lower().endswith(VALID_IMAGE_EXTENSIONS):
                continue
            name_no_ext = os.path.splitext(fname)[0]
            label_path = find_label_path(name_no_ext, folder)
            image_id = f"{folder}/{name_no_ext}"
            records.append((folder, os.path.join(folder_path, fname), label_path, image_id))
    return records


def safe_filename(image_id: str) -> str:
    """'folder/name' -> 'folder__name' so it can be used inside a file name."""
    return image_id.replace("/", "__").replace(os.sep, "__")


image_records = discover_images()
missing_labels = [r for r in image_records if r[2] is None]
valid_images = [r for r in image_records if r[2] is not None]

print(f"Discovered {len(image_records)} images in "
      f"{len(set(r[0] for r in image_records))} folders.")
print(f"With labels: {len(valid_images)} | without labels (skipped): {len(missing_labels)}")
if missing_labels:
    print("  first missing:", [r[3] for r in missing_labels[:5]])


In [ ]:
# =============================================================================
# CELL 8 - IoU
# =============================================================================
# Unchanged from the original notebook - it was already correct and vectorised.
# =============================================================================

def compute_iou_matrix(boxes1, boxes2):
    """
    Pairwise IoU between two sets of [x1, y1, x2, y2] boxes.

    Input : boxes1 (N, 4), boxes2 (M, 4)
    Output: (N, M) matrix, entry [i, j] = IoU(boxes1[i], boxes2[j])
    """
    boxes1 = np.asarray(boxes1, dtype=np.float32).reshape(-1, 4)
    boxes2 = np.asarray(boxes2, dtype=np.float32).reshape(-1, 4)
    if len(boxes1) == 0 or len(boxes2) == 0:
        return np.zeros((len(boxes1), len(boxes2)), dtype=np.float32)

    x1 = np.maximum(boxes1[:, None, 0], boxes2[None, :, 0])   # left
    y1 = np.maximum(boxes1[:, None, 1], boxes2[None, :, 1])   # top
    x2 = np.minimum(boxes1[:, None, 2], boxes2[None, :, 2])   # right
    y2 = np.minimum(boxes1[:, None, 3], boxes2[None, :, 3])   # bottom

    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])
    area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])
    union = area1[:, None] + area2[None, :] - inter
    return np.where(union > 0, inter / union, 0.0).astype(np.float32)

# How much of this predicted bounding box lies inside a certain tile or region?
# how many pixels of a bounding box overlap with another rectangular region
# only intersection area not oc=ver union
def intersection_area(box, region):
    """Plain intersection AREA (not IoU) between one box and one region."""
    x1 = max(box[0], region[0]); y1 = max(box[1], region[1]) # top and left of inter
    x2 = min(box[2], region[2]); y2 = min(box[3], region[3]) # bottom and riht of inter
    return max(0.0, x2 - x1) * max(0.0, y2 - y1) # area of inter = inter width x inter height


print("IoU helpers ready.")

In [ ]:
# =============================================================================
# CELL 9 - CORRECT ONE-TO-ONE MATCHING
# =============================================================================
# THE BUG IN THE OLD CODE (both E02_1 and E02_2 originally shared this bug)
# The old matcher did:
#     best_gt = argmax(IoU[pred])          <- global best, ignoring availability
#     if best_iou >= thr and best_gt free: match
# So if the globally best GT was already taken, the prediction was thrown away
# even when ANOTHER, still free GT had IoU >= threshold. That under-counted TPs
# (recall too low, false positives too high) on images with clustered plants.
#
# THE CORRECT RULE (implemented here, same as E02_2)
#   sort predictions by confidence, high -> low
#   for each prediction:
#       look ONLY at GT boxes that are still unmatched
#       take the unmatched GT with the highest IoU
#       if that IoU >= evaluation IoU threshold -> match, else leave unmatched
#   one GT can be matched by at most one prediction, and vice versa.
#
# This single function is used for TP / FP / FN / precision / recall / F1 /
# IoU1 / IoU2 / confusion matrices, so the whole thesis uses one definition.
# =============================================================================

def match_one_to_one(pred_boxes, pred_scores, gt_boxes, iou_threshold=EVAL_IOU_THRESHOLD):
    """
    Input : pred_boxes  (P, 4), pred_scores (P,), gt_boxes (G, 4), iou_threshold
    Output: dict with
        'pred_match_gt' (P,) int   - matched GT index per prediction, -1 if unmatched
        'gt_match_pred' (G,) int   - matched prediction index per GT,  -1 if unmatched
        'pred_iou'      (P,) float - IoU of the accepted match, 0.0 if unmatched
        'matched_ious'  list       - IoU values of all accepted matches
    """
    pred_boxes = np.asarray(pred_boxes, dtype=np.float32).reshape(-1, 4)
    gt_boxes = np.asarray(gt_boxes, dtype=np.float32).reshape(-1, 4)
    n_pred, n_gt = len(pred_boxes), len(gt_boxes)

    pred_match_gt = np.full(n_pred, -1, dtype=np.int64)
    gt_match_pred = np.full(n_gt, -1, dtype=np.int64)
    pred_iou = np.zeros(n_pred, dtype=np.float32)  # IoU of the accepted match for every prediction

    if n_pred == 0 or n_gt == 0:
        return {"pred_match_gt": pred_match_gt, "gt_match_pred": gt_match_pred,
                "pred_iou": pred_iou, "matched_ious": []}

    iou = compute_iou_matrix(pred_boxes, gt_boxes)
    gt_free = np.ones(n_gt, dtype=bool)  # track which GTs are still available

    # 'stable' keeps the original order for equal scores -> fully deterministic.
    order = np.argsort(-np.asarray(pred_scores, dtype=np.float32), kind="stable")  # confidence, high -> low

    matched_ious = []
    for p in order:
        if not gt_free.any():
            break                                   # nothing left to match
        # Consider ONLY currently unmatched GT boxes (this is the fix).
        candidate_ious = np.where(gt_free, iou[p], -1.0)
        g = int(np.argmax(candidate_ious))  # best FREE GT
        if candidate_ious[g] >= iou_threshold:
            gt_free[g] = False
            pred_match_gt[p] = g
            gt_match_pred[g] = p
            pred_iou[p] = candidate_ious[g]
            matched_ious.append(float(candidate_ious[g]))

    return {"pred_match_gt": pred_match_gt, "gt_match_pred": gt_match_pred,
            "pred_iou": pred_iou, "matched_ious": matched_ious}


def safe_f1(precision, recall):
    """F1 = 2PR/(P+R) with a safe zero denominator (returns 0.0)."""
    denom = precision + recall
    return float(2.0 * precision * recall / denom) if denom > 0 else 0.0


print("One-to-one matcher ready (unmatched-GT-aware).")


In [ ]:
# =============================================================================
# CELL 10 - TILES: GENERATED ONCE PER IMAGE
# =============================================================================
# Open the image once -> build the tile list once -> reuse it for the single run
# of this image. Tiles are kept as CPU/PIL images (never on the GPU).
# Unchanged logic from k_exemplars_with_tiling.
# =============================================================================

def tile_bboxes(img_w, img_h, tile_size=TILE_SIZE, overlap=OVERLAP):
    """
    Sliding-window tiles covering the whole image with overlap, so a plant lying
    on a tile border is fully visible in at least one window.
    Output: list of (x1, y1, x2, y2) in ORIGINAL image coordinates.
    """
    step = max(1, tile_size - overlap)
    tiles = []
    for y in range(0, img_h, step):
        for x in range(0, img_w, step):
            x2 = min(x + tile_size, img_w)
            y2 = min(y + tile_size, img_h)
            x1 = max(0, x2 - tile_size)
            y1 = max(0, y2 - tile_size)
            tiles.append((x1, y1, x2, y2))
    return list(dict.fromkeys(tiles))   # remove duplicates, keep order


def build_tile_cache(image, use_tiling=USE_TILING,
                     tile_size=TILE_SIZE, overlap=OVERLAP,
                     cache_in_memory=CACHE_TILES_IN_MEMORY):
    """
    Build the tile list for ONE already-open image.

    Input : image - PIL image of the full UAV photo
    Output: list of dicts, one per tile:
            {'tile_id', 'x1', 'y1', 'x2', 'y2', 'image' (PIL or None)}
            'image' is None when cache_in_memory=False (cropped on demand).

    If use_tiling is False the whole image is returned as a single "tile", which
    keeps the rest of the pipeline identical for the no-tiling experiment.
    """
    if use_tiling:
        coords = tile_bboxes(image.width, image.height, tile_size, overlap)
    else:
        coords = [(0, 0, image.width, image.height)]

    cache = []
    for tid, (x1, y1, x2, y2) in enumerate(coords):
        cache.append({
            "tile_id": tid, "x1": x1, "y1": y1, "x2": x2, "y2": y2,
            "image": image.crop((x1, y1, x2, y2)) if cache_in_memory else None,
        })
    return cache


def get_tile_image(tile, full_image):
    """Return the tile's PIL image, cropping on demand if it was not cached."""
    if tile["image"] is not None:
        return tile["image"]
    return full_image.crop((tile["x1"], tile["y1"], tile["x2"], tile["y2"]))


print(f"Tiling helpers ready (TILE_SIZE={TILE_SIZE}, OVERLAP={OVERLAP}).")

In [ ]:
# =============================================================================
# CELL 11 - LOCAL BACKGROUND PATCH  (bug fixed)
# =============================================================================
# WHY THE STRIP HAS A TEXTURED BACKGROUND
# The exemplar crops are pasted into a strip above the tile. A flat white strip
# would create a hard artificial edge that the image encoder's self-attention can
# latch onto, and the exemplar feature would describe "a plant in a void" instead
# of "a plant in grass". Sampling real texture from the same tile keeps colour,
# brightness and grain consistent with the current lighting conditions.
#
# THE BUG
# The old version cropped the TOP-LEFT corner of the tile - exactly the rows that
# are then pasted directly ABOVE the tile. The same texture therefore appeared
# twice, immediately adjacent to itself: an obvious repeating seam.
#
# THE FIX
#   - sample from a RANDOM offset inside the tile, deliberately skipping the top
#     band that would sit next to its own copy
#   - the offset comes from a deterministic RNG (stable_seed) -> reproducible
#   - apply a light Gaussian blur so leftover structure does not look like a
#     second, sharp copy of real plants
# =============================================================================

def get_local_background_patch(tile_img, patch_w, patch_h, rng,
                               blur_radius=BACKGROUND_BLUR_RADIUS):
    """
    Input : tile_img        - PIL image of the current tile
            patch_w/patch_h - required size of the strip background
            rng             - np.random.Generator (deterministically seeded)
    Output: PIL image of size (patch_w, patch_h) with local-looking texture.
    """
    tw, th = tile_img.size
    crop_w = min(patch_w, tw)
    crop_h = min(patch_h, th)

    max_x0 = tw - crop_w
    max_y0 = th - crop_h
    # Skip the first patch_h rows: they are the ones that end up directly below
    # the strip, so re-using them would create the adjacent duplication.
    min_y0 = min(patch_h, max_y0)

    x0 = int(rng.integers(0, max_x0 + 1)) if max_x0 > 0 else 0
    y0 = int(rng.integers(min_y0, max_y0 + 1)) if max_y0 > min_y0 else max_y0

    patch = tile_img.crop((x0, y0, x0 + crop_w, y0 + crop_h))
    if patch.size != (patch_w, patch_h):
        patch = patch.resize((patch_w, patch_h), Image.BILINEAR)
    if blur_radius and blur_radius > 0:
        patch = patch.filter(ImageFilter.GaussianBlur(radius=blur_radius))
    return patch


def make_feather_mask(size, feather_width=FEATHER_WIDTH):
    """
    Soft alpha mask so a pasted exemplar crop blends into the strip background
    instead of showing a hard rectangular border. Unchanged from the original.

    Input : size (w, h) of the crop, feather_width in px
    Output: PIL 'L' image used as the paste mask
    """
    w, h = size
    mask = np.full((h, w), 255.0, dtype=np.float32)     # start fully opaque
    effective = min(feather_width, h // 2, w // 2)
    if effective >= 1:
        for i in range(effective):
            alpha = 255.0 * (i + 1) / effective
            mask[i, :] = np.minimum(mask[i, :], alpha)                   # top
            mask[h - 1 - i, :] = np.minimum(mask[h - 1 - i, :], alpha)   # bottom
            mask[:, i] = np.minimum(mask[:, i], alpha)                   # left
            mask[:, w - 1 - i] = np.minimum(mask[:, w - 1 - i], alpha)   # right
    return Image.fromarray(mask.astype(np.uint8), mode="L")


print("Background sampling + feather mask ready.")

In [ ]:
# =============================================================================
# CELL 12 - COMPOSE  (exemplar strip on top + real tile below)  (bug fixed)
# =============================================================================
# THE BUG
# The old code computed
#     canvas_w = max(tile_w, strip_content_w)
# With 3 large exemplars the strip was wider than the tile, so the canvas was
# wider than the tile. Image.new("RGB", ...) fills new pixels with BLACK, and the
# area to the RIGHT of the pasted tile stayed black -> a large artificial black
# rectangle inside the model input.
#
# THE FIX (simple and robust)
#   canvas width is ALWAYS exactly the tile width, so nothing can remain unpainted
#   next to the tile. If the exemplars do not fit in that width, ALL crops are
#   scaled down by ONE common factor:
#       scale = available_width / total_crop_width
#   Using a single common factor preserves each crop's aspect ratio AND the
#   relative size differences between the exemplars. The UAV tile itself is never
#   resized or distorted. Works for 1 and for 3 exemplars.
# =============================================================================

def compose_tile_with_exemplars(tile_img, crop_images, rng,
                                margin=STRIP_MARGIN,
                                feather_width=FEATHER_WIDTH,
                                max_strip_height_fraction=MAX_STRIP_HEIGHT_FRACTION):
    """
    Input : tile_img    - PIL image of the tile
            crop_images - list of PIL exemplar crops (1 or 3)
            rng         - deterministic np.random.Generator for the background
    Output: composed  - PIL image actually sent to SAM3
            crop_boxes- list of [x1,y1,x2,y2] of each exemplar in COMPOSED coords
                        (these are the positive visual prompts)
            offset    - (dx, dy) where the real tile starts inside 'composed'
    """
    n = len(crop_images)
    canvas_w = tile_img.width                       # never wider than the tile
    available_w = canvas_w - margin * (n + 1)       # space left for the crops
    total_crop_w = sum(c.width for c in crop_images)

    # --- 1) shrink the exemplars (aspect ratio preserved) if they do not fit ---
    scale = 1.0
    if total_crop_w > 0 and available_w > 0 and total_crop_w > available_w:
        scale = available_w / float(total_crop_w)

    # --- 2) optional additional height clamp (disabled by default) -------------
    if max_strip_height_fraction is not None:
        max_crop_h = max(1.0, max_strip_height_fraction * tile_img.height - 2 * margin)
        tallest = max(c.height for c in crop_images)
        if tallest * scale > max_crop_h:
            scale = min(scale, max_crop_h / float(tallest))

    if scale < 1.0:
        crop_images = [
            c.resize((max(1, int(round(c.width * scale))),
                      max(1, int(round(c.height * scale)))), Image.BILINEAR)
            for c in crop_images
        ]

    strip_h = max(c.height for c in crop_images) + 2 * margin
    canvas_h = strip_h + tile_img.height

    # --- 3) paint the whole strip with local texture (no unpainted pixels) -----
    composed = Image.new("RGB", (canvas_w, canvas_h))
    composed.paste(get_local_background_patch(tile_img, canvas_w, strip_h, rng), (0, 0))

    # --- 4) paste the real tile below the strip; it fills the full canvas width -
    offset = (0, strip_h)
    composed.paste(tile_img, offset)

    # --- 5) paste the exemplars side by side, with feathered edges -------------
    crop_boxes = []
    cursor_x = margin
    for crop in crop_images:
        composed.paste(crop, (cursor_x, margin), make_feather_mask(crop.size, feather_width))
        crop_boxes.append([cursor_x, margin, cursor_x + crop.width, margin + crop.height])
        cursor_x += crop.width + margin

    return composed, crop_boxes, offset


print("Exemplar strip composition ready (canvas width == tile width, no black padding).")

In [ ]:
# =============================================================================
# CELL 13 - TARGET-REGION FILTER (bug fixed) + PLAUSIBILITY FILTER
# =============================================================================
# THE BUG IN keep_only_target_region_detections
# The old rule was "keep the box only if y1 >= dy - 5", i.e. it looked ONLY at the
# top edge of the box. A real Rumex plant sitting at the very top of a tile whose
# predicted box leaks a few dozen pixels into the exemplar strip was deleted, even
# though almost all of the box was inside the tile.
#
# THE FIX - explicit geometric criterion
# The composed image contains two regions:
#     strip region : y in [0, dy)
#     tile  region : y in [dy, dy + tile_h)
# A detection is kept if at least TILE_REGION_MIN_FRACTION (= 0.50) of its AREA
# lies inside the TILE region, i.e. the box belongs to whichever region holds the
# majority of it. Kept boxes are then CLIPPED to the tile region and shifted into
# tile coordinates. No other heuristic is applied here.
# =============================================================================

def keep_only_target_region_detections(boxes, scores, fill_ratios,
                                       offset, tile_w, tile_h,
                                       min_fraction_inside=TILE_REGION_MIN_FRACTION):
    """
    Input : boxes (N,4) in COMPOSED coordinates, scores (N,), fill_ratios (N,)
            offset (dx, dy) = where the tile starts inside the composed image
            tile_w, tile_h  = size of the real tile
    Output: boxes in TILE coordinates, scores, fill_ratios (all filtered)
    """
    dx, dy = offset
    tile_region = (dx, dy, dx + tile_w, dy + tile_h)

    kept_boxes, kept_scores, kept_fills = [], [], []
    for box, score, fill in zip(boxes, scores, fill_ratios):
        x1, y1, x2, y2 = [float(v) for v in box]
        box_area = max(0.0, x2 - x1) * max(0.0, y2 - y1)
        if box_area <= 0:
            continue
        # fraction of the predicted box that lies inside the real tile
        fraction_inside = intersection_area((x1, y1, x2, y2), tile_region) / box_area
        if fraction_inside < min_fraction_inside:
            continue                                   # belongs to the strip
        # clip to the tile region, then convert composed -> tile coordinates
        cx1 = min(max(x1, tile_region[0]), tile_region[2]) - dx
        cy1 = min(max(y1, tile_region[1]), tile_region[3]) - dy
        cx2 = min(max(x2, tile_region[0]), tile_region[2]) - dx
        cy2 = min(max(y2, tile_region[1]), tile_region[3]) - dy
        kept_boxes.append([cx1, cy1, cx2, cy2])
        kept_scores.append(float(score))
        kept_fills.append(float(fill))
    return kept_boxes, kept_scores, kept_fills


def filter_implausible_boxes(boxes, scores, fill_ratios, tile_w, tile_h,
                             min_fill_ratio=MIN_FILL_RATIO,
                             max_area_fraction=MAX_AREA_FRACTION,
                             edge_margin=EDGE_MARGIN):
    """
    Remove detections that cannot be a single Rumex plant. SAME thresholds and
    SAME logic as the original notebook; the only change is that the mask-fill
    ratio is received as a PRE-COMPUTED number instead of a stored mask, because
    masks must never be kept (KEEP_MASKS = False).

    Input : boxes in TILE coordinates, scores, fill_ratios, tile size
    Output: the surviving boxes / scores / fill_ratios
    """
    kept_boxes, kept_scores, kept_fills = [], [], []
    tile_area = float(tile_w * tile_h)
    for box, score, fill in zip(boxes, scores, fill_ratios):
        x1, y1, x2, y2 = box
        bw, bh = x2 - x1, y2 - y1
        if bw <= edge_margin or bh <= edge_margin:          # tiny boxes
            continue
        if (bw * bh) / tile_area > max_area_fraction:       # absurdly large boxes
            continue
        if fill < min_fill_ratio:                           # hollow boxes
            continue
        kept_boxes.append(box)
        kept_scores.append(score)
        kept_fills.append(fill)
    return kept_boxes, kept_scores, kept_fills


print("Target-region filter (>=50% area inside tile) + plausibility filter ready.")

In [ ]:
# =============================================================================
# CELL 14 - DIVERSITY-BASED EXEMPLAR SELECTION (D1 / D2 / D3)
# =============================================================================
# This is the ONLY methodological change with respect to k_size_exemplars_no_tiling.
#
# STEP 1  crop every GT Rumex of the image, at FULL resolution:
#
#           GT bbox -> +10% fixed context -> square padding -> 224x224 -> DINOv2
#
#         a) the GT box is enlarged by CROP_CONTEXT (10%) of its own width on the
#            left/right and of its own height on the top/bottom, so every plant is
#            embedded together with a little of its surrounding soil/grass, in
#            proportion to its own size;
#         b) that context box is cut out of the image and PASTED, centred, onto a
#            SQUARE canvas of side max(context_w, context_h) filled with
#            CROP_PAD_FILL. Padding (instead of widening the crop) means the extra
#            area is neutral filler rather than more of the field, so the embedding
#            cannot be driven by whatever happens to lie next to an elongated plant.
#            Parts of the context box that fall OUTSIDE the image are padded the same
#            way, so a plant at the image border stays centred in its canvas;
#         c) the square canvas is resized DIRECTLY to 224x224 (aspect ratio of the
#            context box is preserved by the padding, so nothing is stretched).
#            The DINOv2 processor's default "shortest edge 256 + centre-crop 224" is
#            switched OFF (DINOV2_CENTER_CROP = False), otherwise it would crop the
#            context margin away again.
#
# STEP 2  embed each crop with DINOv2 (CLS token of the last hidden layer),
#         then L2-normalise:  z_i = f(crop_i) / ||f(crop_i)||
#
# STEP 3  pairwise cosine distance
#             d(i,j) = 1 - cos(z_i, z_j) = 1 - <z_i, z_j>      (in [0, 2])
#
# STEP 4  diversity-based selection
#             S* = argmax_{|S| = 3}  min_{i,j in S, i != j} d(i,j)
#         "pick the three plants such that even the CLOSEST pair among the three
#          is still as different as possible" (max-min / maximin dispersion).
#         The search is EXHAUSTIVE over all C(G,3) triples, evaluated vectorised,
#         so the optimum is exact - not a greedy approximation. Triples are
#         enumerated in lexicographic order and the FIRST maximiser is taken,
#         so ties are broken by the lowest GT indices -> fully deterministic.
#         (For G > DIVERSITY_EXACT_MAX_GT a greedy farthest-point fallback keeps
#          the cell from exploding; the dataset never reaches that size.)
#
# ROLE ORDER  D1, D2, D3 = the three selected plants sorted by DECREASING mean
#         distance to the other two selected plants (ties -> lower GT index).
#         D1 is therefore the most "isolated" of the three. The order only
#         affects labels/colours and the order in which the boxes are handed to
#         SAM3; the prompt set itself is order-independent.
#
# Fewer than 3 GT boxes: 1 box -> [D1], 2 boxes -> [D1, D2] (both used),
# exactly like the S / S+L fallback of the size-based experiment.
# =============================================================================

def box_areas(boxes):
    """(N,4) [x1,y1,x2,y2] -> (N,) areas in px^2."""
    b = np.asarray(boxes, dtype=np.float32).reshape(-1, 4)
    return (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])


# ----------------------------- STEP 1: cropping -------------------------------
def crop_gt_for_embedding(image, box, context=CROP_CONTEXT, pad_fill=CROP_PAD_FILL):
    """
    GT bbox -> +context -> square padding.  (The resize to 224 is done by the
    DINOv2 processor in embed_gt_crops.)

    image : PIL.Image at ORIGINAL resolution
    box   : [x1, y1, x2, y2] in full-resolution pixels
    Returns a SQUARE PIL.Image (RGB) with the padded context box centred in it.
    """
    img_w, img_h = image.size
    x1, y1, x2, y2 = [float(v) for v in box]
    w = max(1.0, x2 - x1)
    y_h = max(1.0, y2 - y1)

    # (a) +context on every side, proportional to the box itself
    left   = int(np.floor(x1 - context * w))
    right  = int(np.ceil (x2 + context * w))
    top    = int(np.floor(y1 - context * y_h))
    bottom = int(np.ceil (y2 + context * y_h))
    crop_w, crop_h = max(1, right - left), max(1, bottom - top)

    # the part of the context box that actually exists inside the image
    ax1, ay1 = max(left, 0), max(top, 0)
    ax2, ay2 = min(right, img_w), min(bottom, img_h)
    if ax2 <= ax1 or ay2 <= ay1:                    # degenerate box outside the image
        return Image.new("RGB", (DINOV2_INPUT_SIZE, DINOV2_INPUT_SIZE),
                         (124, 116, 104) if pad_fill == "edge" else pad_fill)
    patch_img = image.crop((ax1, ay1, ax2, ay2))

    # (b) paste it, centred, on a square canvas -> padding, never stretching
    side = max(crop_w, crop_h)
    off_x = (side - crop_w) // 2 + (ax1 - left)     # keeps the plant centred even at
    off_y = (side - crop_h) // 2 + (ay1 - top)      # the image border

    if pad_fill == "edge":                          # replicate border pixels
        canvas = patch_img.resize((side, side), Image.NEAREST)
        canvas.paste(patch_img, (off_x, off_y))
    else:
        canvas = Image.new("RGB", (side, side), tuple(pad_fill))
        canvas.paste(patch_img, (off_x, off_y))
    return canvas


# ---------------------- STEP 2: DINOv2 embeddings -----------------------------
@torch.no_grad()
def embed_gt_crops(image, gt_boxes, batch_size=DINOV2_BATCH_SIZE):
    """
    image    : PIL.Image, ORIGINAL resolution
    gt_boxes : (G,4) full-resolution GT boxes
    Returns  : (G, D) float32 array of L2-NORMALISED DINOv2 CLS embeddings.
    Pipeline : GT bbox -> +CROP_CONTEXT -> square padding -> 224x224 -> DINOv2 -> CLS.
    """
    crops = [crop_gt_for_embedding(image, b) for b in np.asarray(gt_boxes).reshape(-1, 4)]
    chunks = []
    for start in range(0, len(crops), batch_size):
        batch = crops[start:start + batch_size]
        # explicit 224x224 resize, NO centre crop -> the +10% context survives
        inputs = dino_processor(
            images=batch,
            do_resize=True,
            size={"height": DINOV2_INPUT_SIZE, "width": DINOV2_INPUT_SIZE},
            do_center_crop=DINOV2_CENTER_CROP,
            return_tensors="pt",
        ).to(dino_model.device)
        if DINOV2_DTYPE != torch.float32:
            inputs["pixel_values"] = inputs["pixel_values"].to(DINOV2_DTYPE)
        outputs = dino_model(**inputs)
        if EMBEDDING_FEATURE == "cls":
            feats = outputs.last_hidden_state[:, 0, :]          # CLS token
        else:
            raise ValueError(f"EMBEDDING_FEATURE={EMBEDDING_FEATURE!r} not implemented.")
        chunks.append(feats.float().cpu().numpy())
        del inputs, outputs, feats

    z = np.concatenate(chunks, axis=0).astype(np.float32)
    norms = np.linalg.norm(z, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return z / norms                                           # ||z_i|| = 1


# ------------------- STEP 3: pairwise cosine distances ------------------------
def cosine_distance_matrix(z):
    """z (G,D) L2-normalised -> (G,G) distances d(i,j) = 1 - <z_i, z_j>, diag = 0."""
    d = 1.0 - (np.asarray(z, dtype=np.float32) @ np.asarray(z, dtype=np.float32).T)
    d = np.clip(d, 0.0, 2.0)
    d = 0.5 * (d + d.T)                    # exact symmetry despite float round-off
    np.fill_diagonal(d, 0.0)
    return d.astype(np.float32)


# --------------- STEP 4: max-min (maximin dispersion) selection ---------------
def _maxmin_exhaustive(d, k):
    """Exact argmax over all C(n,k) subsets of the minimum in-subset distance."""
    n = d.shape[0]
    combos = np.fromiter(itertools.chain.from_iterable(itertools.combinations(range(n), k)),
                         dtype=np.int64).reshape(-1, k)             # lexicographic order
    pair_mins = None
    for a, b in itertools.combinations(range(k), 2):
        dist_ab = d[combos[:, a], combos[:, b]]
        pair_mins = dist_ab if pair_mins is None else np.minimum(pair_mins, dist_ab)
    best = int(np.argmax(pair_mins))        # first maximiser -> lowest indices on ties
    return [int(i) for i in combos[best]], float(pair_mins[best])


def _maxmin_greedy(d, k):
    """Farthest-point fallback for very large G: start from the farthest pair,
    then repeatedly add the point whose minimum distance to the set is largest."""
    n = d.shape[0]
    i, j = np.unravel_index(int(np.argmax(np.triu(d, 1))), d.shape)
    chosen = [int(i), int(j)]
    while len(chosen) < k:
        min_to_set = d[:, chosen].min(axis=1)
        min_to_set[chosen] = -np.inf
        chosen.append(int(np.argmax(min_to_set)))
    sub = d[np.ix_(chosen, chosen)]
    min_pair = float(sub[np.triu_indices(len(chosen), 1)].min())
    return sorted(chosen), min_pair


def select_diverse_exemplars(gt_boxes, embeddings, k=K_EXEMPLARS):
    """
    Input : gt_boxes   (G,4) full-resolution GT boxes of ONE image
            embeddings (G,D) L2-normalised DINOv2 embeddings of the same boxes
    Output: dict with
              'indices'   - GT indices in prompt order (D1, D2, D3)
              'roles'     - ["D1","D2","D3"] (shorter when G < 3)
              'areas'     - area of each selected box (px^2, reported only)
              'distance_matrix'        - (G,G) pairwise cosine distances
              'min_pairwise_distance'  - the maximised objective  min_{i,j in S} d(i,j)
              'mean_pairwise_distance' - mean distance inside the selected triple
              'gt_mean_distance'       - mean distance over ALL GT pairs of the image
              'search_mode'            - "exhaustive" | "greedy" | "all_gt_used"
              'median_area', 'all_areas'
    """
    areas = box_areas(gt_boxes)
    n = len(areas)
    if n == 0:
        raise ValueError("The image has no GT boxes -> no diversity-based exemplars.")

    d = cosine_distance_matrix(embeddings)
    all_pairs = d[np.triu_indices(n, 1)] if n > 1 else np.array([0.0], dtype=np.float32)

    if n <= k:                                   # 1 or 2 GT boxes -> use them all
        indices = list(range(n))
        min_pair = float(all_pairs.min()) if n > 1 else float("nan")
        search_mode = "all_gt_used"
    elif n <= DIVERSITY_EXACT_MAX_GT:
        indices, min_pair = _maxmin_exhaustive(d, k)
        search_mode = "exhaustive"
    else:
        indices, min_pair = _maxmin_greedy(d, k)
        search_mode = "greedy"

    # ---- role order: decreasing mean distance to the other selected plants ----
    if len(indices) > 1:
        sub = d[np.ix_(indices, indices)]
        mean_to_others = sub.sum(axis=1) / (len(indices) - 1)
        order = sorted(range(len(indices)),
                       key=lambda p: (-float(mean_to_others[p]), indices[p]))
        indices = [indices[p] for p in order]
        sub = d[np.ix_(indices, indices)]
        mean_pair = float(sub[np.triu_indices(len(indices), 1)].mean())
    else:
        mean_pair = float("nan")

    roles = [f"D{r + 1}" for r in range(len(indices))]

    return {
        "indices": indices,
        "roles": roles,
        "areas": areas[indices],
        "distance_matrix": d,
        "min_pairwise_distance": float(min_pair),
        "mean_pairwise_distance": float(mean_pair),
        "gt_mean_distance": float(all_pairs.mean()),
        "search_mode": search_mode,
        "median_area": float(np.median(areas)),
        "all_areas": areas,
    }


def format_diversity_prompt_id(selection):
    """e.g. indices [12, 5, 40] with roles D1,D2,D3 -> 'D1:12+D2:5+D3:40'."""
    return "+".join(f"{r}:{i}" for r, i in zip(selection["roles"], selection["indices"]))


# --- self-test on toy embeddings (no GPU / no model needed) --------------------
# 5 unit vectors: 0, 1, 2 are mutually orthogonal (distance 1 to each other),
# 3 and 4 are near-duplicates of 0 and 1. The max-min triple must be {0, 1, 2}.
_toy_z = np.array([
    [1.0, 0.0, 0.0, 0.0],
    [0.0, 1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0, 0.0],
    [0.9998, 0.02, 0.0, 0.0],
    [0.02, 0.9998, 0.0, 0.0],
], dtype=np.float32)
_toy_z /= np.linalg.norm(_toy_z, axis=1, keepdims=True)
_toy_boxes = np.array([[0, 0, 10, 10]] * 5, dtype=np.float32)
_sel = select_diverse_exemplars(_toy_boxes, _toy_z)
assert sorted(_sel["indices"]) == [0, 1, 2], _sel["indices"]
assert abs(_sel["min_pairwise_distance"] - 1.0) < 1e-4, _sel["min_pairwise_distance"]
# --- self-test of the crop geometry (no GPU / no model needed) -----------------
_probe = Image.new("RGB", (200, 120), (0, 0, 0))
for _box, _expect in [([50, 40, 90, 60], 48),      # 40x20 box -> 48x24 context -> 48x48 canvas
                      ([0, 0, 20, 20], 24),        # touches the corner -> padded, still square
                      ([190, 110, 200, 120], 12)]: # bottom-right corner
    _c = crop_gt_for_embedding(_probe, _box)
    assert _c.size == (_expect, _expect), (_box, _c.size, _expect)
print(f"Crop pipeline ready: GT bbox -> +{CROP_CONTEXT:.0%} context -> square padding "
      f"-> {DINOV2_INPUT_SIZE}x{DINOV2_INPUT_SIZE} -> DINOv2 ({EMBEDDING_FEATURE}).")

print("Diversity-based exemplar selection ready "
      f"({DIVERSITY_OBJECTIVE}, {DIVERSITY_METRIC}). Toy example:",
      format_diversity_prompt_id(_sel),
      f"| min pairwise distance = {_sel['min_pairwise_distance']:.3f} "
      f"| search = {_sel['search_mode']}")


In [ ]:
# =============================================================================
# CELL 15 - SAM3 BATCH INFERENCE  (FP16, no masks kept)
# =============================================================================
# WHAT CHANGED VS THE ORIGINAL
#  * tiles are sent in batches of BATCH_SIZE (4) instead of one at a time
#  * torch.inference_mode() + autocast(float16) instead of torch.no_grad()
#  * masks are converted into ONE number per detection (the mask-fill ratio used
#    by the plausibility filter) and then deleted immediately. Nothing mask-shaped
#    ever leaves this function -> no GPU/RAM/disk cost.
#  * SAM3 is called once per TILE (batched) at CONFIDENCE_THRESHOLD = 0.30
#    (fixed from the start); NMS is applied later, offline.
# =============================================================================

def _to_numpy(x):
    """Torch tensor (any device/dtype) or numpy array -> float32 numpy array."""
    if torch.is_tensor(x):
        return x.detach().float().cpu().numpy()
    return np.asarray(x, dtype=np.float32)


def _fill_ratios_from_masks(boxes_np, masks, binarise_at=0.5):
    """
    Fraction of each predicted box that is actually covered by its mask - the
    single number the plausibility filter needs. Only the small box region is
    materialised; the mask itself is dropped by the caller immediately after.

    Input : boxes_np (N,4) in composed coords,
            masks    (N,H,W) torch tensor OR list of (H,W) masks
    Output: (N,) float32 fill ratios
    """
    n = len(boxes_np)
    fills = np.zeros(n, dtype=np.float32)
    if masks is None or n == 0:
        return fills
    for i in range(n):
        m = masks[i]                        # works for a tensor and for a list
        h, w = int(m.shape[-2]), int(m.shape[-1])
        x1, y1, x2, y2 = boxes_np[i]
        x1c, y1c = int(max(0, np.floor(x1))), int(max(0, np.floor(y1)))
        x2c, y2c = int(min(w, np.ceil(x2))), int(min(h, np.ceil(y2)))
        if x2c <= x1c or y2c <= y1c:
            continue
        region = m[..., y1c:y2c, x1c:x2c]
        if torch.is_tensor(region):
            if region.dtype == torch.bool:
                fills[i] = float(region.float().mean())
            else:
                fills[i] = float((region > binarise_at).float().mean())
        else:
            region = np.asarray(region)
            fills[i] = float((region > binarise_at).mean()) if region.size else 0.0
    return fills


def sam3_infer_batch(composed_images, crop_boxes_batch,
                     threshold=CONFIDENCE_THRESHOLD):
    """
    Run SAM3 on a batch of composed images (exemplar strip + tile).

    Input : composed_images  - list of PIL images (length <= BATCH_SIZE)
            crop_boxes_batch - list (same length) of lists of exemplar boxes,
                               each [x1,y1,x2,y2] in that composed image
    Output: list (same length) of (boxes, scores, fill_ratios), all numpy,
            boxes in COMPOSED-image coordinates, every detection with
            score >= threshold. Masks are NOT returned.
    """
    inputs = sam3_processor(
        images=composed_images,
        input_boxes=[[[float(v) for v in b] for b in boxes] for boxes in crop_boxes_batch],
        input_boxes_labels=[[1] * len(boxes) for boxes in crop_boxes_batch],  # 1 = positive
        return_tensors="pt",
    ).to(sam3_model.device)

    # match the pixel tensor dtype to the (possibly fp16) model weights
    if MODEL_DTYPE == torch.float16 and "pixel_values" in inputs:
        inputs["pixel_values"] = inputs["pixel_values"].to(torch.float16)

    per_image = []
    with torch.inference_mode():
        if MODEL_DTYPE == torch.float16:
            with torch.autocast("cuda", dtype=torch.float16):
                outputs = sam3_model(**inputs)
        else:
            outputs = sam3_model(**inputs)

        # target_sizes -> predictions are mapped back to the composed-image size
        # (SAM3 internally resizes the input to its own resolution).
        results = sam3_processor.post_process_instance_segmentation(
            outputs,
            threshold=threshold,
            mask_threshold=MASK_THRESHOLD,
            target_sizes=inputs.get("original_sizes").tolist(),
        )

        for i, res in enumerate(results):
            boxes = _to_numpy(res["boxes"]).reshape(-1, 4)
            scores = _to_numpy(res["scores"]).reshape(-1)
            fills = _fill_ratios_from_masks(boxes, res.get("masks", None))
            # masks are no longer needed -> free them immediately
            if "masks" in res:
                res["masks"] = None
            per_image.append((boxes, scores, fills))

    del inputs, outputs, results
    return per_image


def run_exemplars_over_tiles(tile_cache, full_image, exemplar_crops, run_seed,
                          batch_size=BATCH_SIZE, threshold=CONFIDENCE_THRESHOLD):
    """
    Run SAM3 for ONE exemplar set (the 3 diverse plants of this image) over
    ALL cached tiles of that image.

    Input : tile_cache     - list of tile dicts (CELL 10), built once per image
            full_image     - the open PIL image (used only if tiles are not cached)
            exemplar_crops - list of PIL crops used as visual prompts
                             (the padded square crops of CELL 14)
            run_seed       - deterministic seed for the strip background sampling
    Output: dict of numpy arrays, all in ORIGINAL-IMAGE coordinates:
            boxes (N,4), scores (N,), fill_ratio (N,), tile_id (N,),
            tile_boxes (N,4)  <- the tile each detection came from
            These are the PRE-NMS detections (score >= 0.30, no NMS) that get
            written to disk for the offline evaluation.
    """
    all_boxes, all_scores, all_fills, all_tids, all_tboxes = [], [], [], [], []

    for start in range(0, len(tile_cache), batch_size):
        batch_tiles = tile_cache[start:start + batch_size]

        composed_images, crop_boxes_batch, offsets, sizes = [], [], [], []
        for tile in batch_tiles:
            tile_img = get_tile_image(tile, full_image)
            # deterministic RNG per (run, tile) -> identical strip background
            # every time this run is repeated
            rng = np.random.default_rng(stable_seed(run_seed, "tile", tile["tile_id"]))
            composed, crop_boxes, offset = compose_tile_with_exemplars(
                tile_img, exemplar_crops, rng)
            composed_images.append(composed)
            crop_boxes_batch.append(crop_boxes)
            offsets.append(offset)
            sizes.append((tile_img.width, tile_img.height))

        batch_results = sam3_infer_batch(composed_images, crop_boxes_batch, threshold)

        for tile, (boxes, scores, fills), offset, (tw, th) in zip(
                batch_tiles, batch_results, offsets, sizes):
            # 1) keep only detections that belong to the real tile, remap to tile coords
            boxes, scores, fills = keep_only_target_region_detections(
                boxes, scores, fills, offset, tw, th)
            # 2) existing plausibility filter (unchanged thresholds)
            boxes, scores, fills = filter_implausible_boxes(boxes, scores, fills, tw, th)
            # 3) tile coords -> original image coords
            for b, s, f in zip(boxes, scores, fills):
                all_boxes.append([b[0] + tile["x1"], b[1] + tile["y1"],
                                  b[2] + tile["x1"], b[3] + tile["y1"]])
                all_scores.append(float(s))
                all_fills.append(float(f))
                all_tids.append(int(tile["tile_id"]))
                all_tboxes.append([tile["x1"], tile["y1"], tile["x2"], tile["y2"]])

        del composed_images, crop_boxes_batch, batch_results

    return {
        "boxes": np.array(all_boxes, dtype=np.float32).reshape(-1, 4),
        "scores": np.array(all_scores, dtype=np.float32).reshape(-1),
        "fill_ratio": np.array(all_fills, dtype=np.float32).reshape(-1),
        "tile_id": np.array(all_tids, dtype=np.int32).reshape(-1),
        "tile_boxes": np.array(all_tboxes, dtype=np.int32).reshape(-1, 4),
    }


print("SAM3 batched inference helpers ready.")

In [ ]:
# =============================================================================
# CELL 16 - PRE-NMS DETECTION STORAGE
# =============================================================================
# For every run (= one image, one D1/D2/D3 prompt set) we store the detections
#   AFTER SAM3 inference at CONFIDENCE_THRESHOLD -> target-region filtering ->
#   plausibility filtering -> conversion to original-image coordinates,
# but BEFORE NMS.
# The offline evaluation (CELL 18 onwards) therefore never needs SAM3 again.
#
# Each detection keeps the tile it came from (tile_id / tile_boxes), so the NMS
# of CELL 19 can report how many duplicates were cross-tile. Masks are never
# stored. The DINOv2 distance matrix of the image IS stored (G x G float32, a few
# kB), so the selection can be re-analysed without re-running DINOv2.
# =============================================================================

def run_npz_path(image_id):
    """Path of the NPZ holding the pre-NMS detections of one image."""
    return os.path.join(RAW_DETECTIONS_DIR, f"{safe_filename(image_id)}.npz")


def save_run_detections(image_id, detections, gt_boxes, selection, image_size, n_tiles):
    """
    Write one run's pre-NMS detections to NPZ. The file is self-contained: it also
    stores the GT boxes, the D1/D2/D3 prompt indices (in prompt order), the tile
    provenance and the pairwise DINOv2 distance matrix, so the whole offline
    evaluation and the final plot can run without re-opening label files, tiles
    or re-embedding the crops.
    """
    path = run_npz_path(image_id)
    np.savez_compressed(
        path,
        experiment_name=np.array(EXPERIMENT_NAME),
        image_ID=np.array(image_id),
        prompt_indices=np.array(selection["indices"], dtype=np.int32),   # order: D1, D2, D3
        prompt_roles=np.array(selection["roles"]),                       # ["D1","D2","D3"]
        prompt_areas=np.asarray(selection["areas"], dtype=np.float32),
        median_gt_area=np.array(float(selection["median_area"]), dtype=np.float32),
        distance_matrix=np.asarray(selection["distance_matrix"], dtype=np.float32),
        min_pairwise_distance=np.array(float(selection["min_pairwise_distance"]), dtype=np.float32),
        mean_pairwise_distance=np.array(float(selection["mean_pairwise_distance"]), dtype=np.float32),
        gt_mean_distance=np.array(float(selection["gt_mean_distance"]), dtype=np.float32),
        search_mode=np.array(selection["search_mode"]),
        n_tiles=np.array(int(n_tiles)),
        image_width=np.array(int(image_size[0])),
        image_height=np.array(int(image_size[1])),
        gt_boxes=gt_boxes.astype(np.float32),
        boxes=detections["boxes"].astype(np.float32),      # x1,y1,x2,y2 (original img)
        scores=detections["scores"].astype(np.float32),    # confidence >= 0.30
        fill_ratio=detections["fill_ratio"].astype(np.float32),
        tile_id=detections["tile_id"].astype(np.int32),    # which tile produced it
        tile_boxes=detections["tile_boxes"].astype(np.int32),  # that tile's extent
    )
    return path


def load_run_detections(path):
    """Read one run NPZ back into a plain python dict."""
    with np.load(path, allow_pickle=False) as z:
        return {
            "image_ID": str(z["image_ID"]),
            "prompt_indices": z["prompt_indices"].astype(int),
            "prompt_roles": [str(r) for r in z["prompt_roles"]],
            "prompt_areas": z["prompt_areas"].reshape(-1),
            "median_gt_area": float(z["median_gt_area"]),
            "distance_matrix": z["distance_matrix"],
            "min_pairwise_distance": float(z["min_pairwise_distance"]),
            "mean_pairwise_distance": float(z["mean_pairwise_distance"]),
            "gt_mean_distance": float(z["gt_mean_distance"]),
            "search_mode": str(z["search_mode"]),
            "n_tiles": int(z["n_tiles"]),
            "image_width": int(z["image_width"]),
            "image_height": int(z["image_height"]),
            "gt_boxes": z["gt_boxes"].reshape(-1, 4),
            "boxes": z["boxes"].reshape(-1, 4),
            "scores": z["scores"].reshape(-1),
            "fill_ratio": z["fill_ratio"].reshape(-1),
            "tile_id": z["tile_id"].reshape(-1),
            "tile_boxes": z["tile_boxes"].reshape(-1, 4),
        }


print("Pre-NMS detection storage ready ->", RAW_DETECTIONS_DIR)


In [ ]:
# =============================================================================
# CELL 17 - MAIN GPU INFERENCE LOOP (DINOv2 -> selection -> tiles -> SAM3)
# =============================================================================
# FOR EACH IMAGE:
#     open the original image ONCE
#     read its GT boxes ONCE
#     crop every GT box (+10% context, square padding) and embed it with DINOv2
#     pick the 3 most DIVERSE GT boxes (max-min cosine distance, CELL 14)
#         - deterministic, so there is exactly ONE run per image
#     build the overlapping tiles ONCE (cached on CPU)
#     run SAM3 over those tiles in batches, each tile composed with the exemplar
#         strip built from the SAME padded crops (CELL 12)
#     save the PRE-NMS detections + tile provenance + distance matrix (NPZ)
#     release the image and the tile cache
#
# Images without any Rumex GT box are skipped, exactly like in the other
# experiments, so every experiment is evaluated on the same set of images.
# NO NMS and NO metric computation happens here - that is all done offline in the
# following cells. The loop is resumable: finished images are listed in
# runs_manifest.csv.
# =============================================================================

# ---- resume support ---------------------------------------------------------
done_images = set()
manifest_exists = os.path.exists(RUN_MANIFEST_CSV)
if manifest_exists:
    _m = pd.read_csv(RUN_MANIFEST_CSV)
    _m = _m[_m["experiment_name"] == EXPERIMENT_NAME]
    other_types = sorted(set(_m["Prompt_Type"].astype(str)) - {PROMPT_TYPE})
    if other_types:
        raise RuntimeError(
            f"{RUN_MANIFEST_CSV} already holds results for another prompt setting "
            f"{other_types}. Use a new EXPERIMENT_NAME for {PROMPT_TYPE} "
            "(or delete the old results folder) so the runs are not mixed.")
    done_images = set(_m["image_ID"].astype(str))
    print(f"Resuming: {len(done_images)} images already finished for {EXPERIMENT_NAME}.")

manifest_file = open(RUN_MANIFEST_CSV, "a", newline="")
manifest_writer = csv.DictWriter(manifest_file, fieldnames=MANIFEST_COLUMNS)
if not manifest_exists:
    manifest_writer.writeheader()

start_time = time.time()
n_new_runs = 0
image_times = []
n_total_images = len(valid_images)

try:
    for img_idx, (folder, image_path, label_path, image_id) in enumerate(valid_images, start=1):
        if image_id in done_images:
            print(f"[{EXPERIMENT_NAME}] ({img_idx}/{n_total_images}) {image_id}: already done, skipped.")
            continue
        image_t0 = time.time()

        # ---------------- open the original image exactly once ----------------
        image = Image.open(image_path).convert("RGB")
        img_w, img_h = image.size
        gt_boxes = load_yolo_boxes(label_path, img_w, img_h, RUMEX_CLASS_ID)
        n_gt = len(gt_boxes)

        if n_gt == 0:
            print(f"[{EXPERIMENT_NAME}] ({img_idx}/{n_total_images}) {image_id}: 0 GT boxes, skipped.")
            image.close(); del image; gc.collect()
            continue

        # ------- DINOv2 embeddings + diversity-based selection (CELL 14) -------
        # full resolution, independent of the tile grid
        embed_t0 = time.time()
        gt_embeddings = embed_gt_crops(image, gt_boxes)
        selection = select_diverse_exemplars(gt_boxes, gt_embeddings)
        embed_seconds = time.time() - embed_t0

        exemplar_indices = selection["indices"]
        prompt_id = format_diversity_prompt_id(selection)

        # the crops pasted into the strip are the SAME padded square crops that
        # produced the embeddings (EXEMPLAR_CROP_FOR_STRIP = "embedding_crop")
        if EXEMPLAR_CROP_FOR_STRIP == "embedding_crop":
            exemplar_crops = [crop_gt_for_embedding(image, gt_boxes[i]) for i in exemplar_indices]
        elif EXEMPLAR_CROP_FOR_STRIP == "tight_gt_box":
            exemplar_crops = [safe_crop(image, gt_boxes[i]) for i in exemplar_indices]
        else:
            raise ValueError(f"Unknown EXEMPLAR_CROP_FOR_STRIP={EXEMPLAR_CROP_FOR_STRIP!r}")

        # ---------------- build the tiles exactly once -------------------------
        tile_cache = build_tile_cache(image)
        n_tiles = len(tile_cache)

        # ---------------- SAM3 over all tiles (CELL 15) ------------------------
        sam_t0 = time.time()
        run_seed = stable_seed(EXPERIMENT_NAME, image_id, "strip_bg")
        detections = run_exemplars_over_tiles(tile_cache, image, exemplar_crops, run_seed)
        sam_seconds = time.time() - sam_t0

        npz_path = save_run_detections(image_id, detections, gt_boxes, selection,
                                       (img_w, img_h), n_tiles)

        areas_by_role = dict(zip(selection["roles"], selection["areas"]))
        run_seconds = time.time() - image_t0
        manifest_writer.writerow({
            "experiment_name": EXPERIMENT_NAME,
            "image_ID": image_id,
            "Prompt_ID": prompt_id,
            "Prompt_Type": PROMPT_TYPE,
            "n_gt": n_gt,
            "n_prompt_gt": len(exemplar_indices),
            "area_D1_px2": round(float(areas_by_role["D1"])) if "D1" in areas_by_role else "",
            "area_D2_px2": round(float(areas_by_role["D2"])) if "D2" in areas_by_role else "",
            "area_D3_px2": round(float(areas_by_role["D3"])) if "D3" in areas_by_role else "",
            "median_gt_area_px2": round(float(selection["median_area"])),
            "min_pairwise_distance": round(float(selection["min_pairwise_distance"]), 6),
            "mean_pairwise_distance": round(float(selection["mean_pairwise_distance"]), 6),
            "gt_mean_distance": round(float(selection["gt_mean_distance"]), 6),
            "search_mode": selection["search_mode"],
            "n_tiles": n_tiles,
            "n_detections_pre_nms": int(len(detections["scores"])),
            "image_width": img_w,
            "image_height": img_h,
            "npz_file": os.path.basename(npz_path),
            "embedding_seconds": round(embed_seconds, 2),
            "inference_seconds": round(run_seconds, 2),
        })
        manifest_file.flush()                  # this image is on disk -> resumable
        n_new_runs += 1

        if len(exemplar_indices) < K_EXEMPLARS:
            print(f"    NOTE: only {n_gt} GT box(es) -> {len(exemplar_indices)} prompt(s) "
                  f"({'+'.join(selection['roles'])}).")

        n_dets = int(len(detections["scores"]))

        # ---------------- release the image and its tile cache -----------------
        for t in tile_cache:
            t["image"] = None
        del tile_cache, detections, exemplar_crops, gt_embeddings
        image.close()
        del image
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()

        image_times.append(time.time() - image_t0)
        avg_per_image = float(np.mean(image_times))
        eta = (n_total_images - img_idx) * avg_per_image
        print(f"[{EXPERIMENT_NAME}] ({img_idx}/{n_total_images}) {image_id} | {n_gt} GT box(es) | "
              f"prompts={prompt_id} | min_dist={selection['min_pairwise_distance']:.3f} "
              f"(mean over all GT pairs {selection['gt_mean_distance']:.3f}) | "
              f"tiles={n_tiles} | pre-NMS detections={n_dets} | "
              f"DINOv2 {embed_seconds:.1f}s + SAM3 {sam_seconds:.1f}s = {run_seconds:.1f}s | "
              f"avg/image={avg_per_image:.1f}s | ETA={eta / 60:.1f} min ({eta / 3600:.2f} h)")
finally:
    manifest_file.close()                      # also closed if the loop crashes

total_elapsed = time.time() - start_time
print(f"\nInference finished for {EXPERIMENT_NAME}: {n_new_runs} new images.")
print(f"Total time: {total_elapsed / 60:.1f} min ({total_elapsed / 3600:.2f} h)")
print(f"Pre-NMS detections in: {RAW_DETECTIONS_DIR}")


In [ ]:
# =============================================================================
# CELL 18 - LOAD CACHED PRE-NMS DETECTIONS
# =============================================================================
# From here on SAM3 is never touched again. Everything below works on the NPZ
# files written in CELL 17, so you can restart the runtime, free the GPU and
# still redo the complete evaluation in a few minutes.
# To run the evaluation only: run CELLS 1-4 and 6-12 (skip CELL 5 and CELL 17),
# then continue from here.
# =============================================================================

manifest = pd.read_csv(RUN_MANIFEST_CSV)
manifest = manifest[(manifest["experiment_name"] == EXPERIMENT_NAME) &
                    (manifest["Prompt_Type"].astype(str) == PROMPT_TYPE)].copy()
manifest = manifest.drop_duplicates(subset=["image_ID"], keep="last")

runs = []
for _, row in manifest.iterrows():
    path = os.path.join(RAW_DETECTIONS_DIR, str(row["npz_file"]))
    if not os.path.exists(path):
        print("MISSING npz (skipped):", path)
        continue
    run = load_run_detections(path)
    run["Prompt_ID"] = str(row["Prompt_ID"])
    run["Prompt_Type"] = str(row["Prompt_Type"])
    runs.append(run)

runs_by_image = {r["image_ID"]: r for r in runs}

print(f"Loaded {len(runs)} runs (= images) for {EXPERIMENT_NAME} ({PROMPT_TYPE}).")
print("Total pre-NMS detections:", int(sum(len(r['scores']) for r in runs)))
print("Total GT boxes:", int(sum(len(r['gt_boxes']) for r in runs)))
print("Tiles per image:", ", ".join(str(x) for x in sorted(set(r['n_tiles'] for r in runs))))
print("Images with fewer than K prompts:",
      int(sum(1 for r in runs if len(r["prompt_indices"]) < K_EXEMPLARS)))

In [ ]:
# =============================================================================
# CELL 19 - OFFLINE NMS
# =============================================================================
# Because the same plant is visible in several overlapping tiles, it can be
# detected several times. NMS keeps the highest-scoring box of each overlapping
# group. The NMS IoU threshold is fixed (NMS_IOU_THRESHOLD = 0.40) and applied
# offline, so the raw pre-NMS detections stay untouched on disk.
#
# 'Provenance' = we also record WHICH detection suppressed which. That lets the
# per-image table (CELL 24) report how many duplicates came from a DIFFERENT
# tile (cross-tile duplication) versus the same tile.
# =============================================================================

def nms_with_provenance(boxes, scores, iou_threshold):
    """
    Input : boxes (N,4), scores (N,), iou_threshold
    Output: keep (list of kept indices, highest score first)
            suppressed (list of (suppressed_index, suppressor_index))
    A detection is suppressed when its IoU with an already kept, higher-scoring
    detection is GREATER than the threshold (same convention as the original code).
    """
    n = len(boxes)
    if n == 0:
        return [], []
    order = list(np.argsort(-np.asarray(scores, dtype=np.float32), kind="stable"))
    keep, suppressed = [], []
    while order:
        i = int(order[0])
        keep.append(i)
        rest = np.array(order[1:], dtype=int)
        if rest.size == 0:
            break
        ious = compute_iou_matrix(boxes[i:i + 1], boxes[rest])[0]
        for s in rest[ious > iou_threshold]:
            suppressed.append((int(s), i))
        order = list(rest[ious <= iou_threshold])
    return keep, suppressed


def apply_nms_to_run(run, iou_threshold):
    """
    Apply NMS to one run's pre-NMS detections.

    Output: dict with 'boxes', 'scores', 'tile_id', 'tile_boxes' of the surviving
            detections SORTED BY SCORE (high -> low), plus the duplicate-source
            counters used later by the diagnostics.
    Sorting by score means that applying an operating confidence threshold later
    is just a prefix selection.
    """
    keep, suppressed = nms_with_provenance(run["boxes"], run["scores"], iou_threshold)
    keep = np.array(keep, dtype=int)

    cross_tile_suppressed, same_tile_suppressed = 0, 0
    for s, k in suppressed:
        if run["tile_id"][s] != run["tile_id"][k]:
            cross_tile_suppressed += 1
        else:
            same_tile_suppressed += 1

    return {
        "boxes": run["boxes"][keep].reshape(-1, 4),
        "scores": run["scores"][keep].reshape(-1),
        "tile_id": run["tile_id"][keep].reshape(-1),
        "tile_boxes": run["tile_boxes"][keep].reshape(-1, 4),
        "n_pre_nms": int(len(run["scores"])),
        "n_suppressed_cross_tile": int(cross_tile_suppressed),
        "n_suppressed_same_tile": int(same_tile_suppressed),
    }


print("Offline NMS ready. NMS IoU threshold (fixed):", NMS_IOU_THRESHOLD)

In [ ]:
# =============================================================================
# CELL 20 - EVALUATION CORE: all_gt AND held_out
# =============================================================================
# all_gt   : classical evaluation, every GT box of the image counts.
#
# held_out : the GT instances that were shown to SAM3 as visual prompts are
#            REMOVED from the GT set, and predictions that fall on those prompt
#            plants are IGNORED (they are neither TP nor FP, they simply do not
#            exist for this evaluation). It answers: "after being shown a few
#            examples, how well does SAM3 find the REMAINING Rumex plants?"
#
# ORDER OF OPERATIONS (important, this is the agreed protocol):
#   1. match predictions to the evaluated (non-prompt) GT with the corrected
#      one-to-one matcher at EVAL_IOU_THRESHOLD = 0.50
#   2. every STILL UNMATCHED prediction whose best IoU with a PROMPT GT box is
#      >= PROMPT_IGNORE_IOU (0.50) becomes IGNORED
#   3. whatever is still unmatched is a false positive
#   Prompt GT boxes themselves are never counted as false negatives.
# =============================================================================

STATUS_FP, STATUS_TP, STATUS_IGNORED = 0, 1, 2


def split_gt_for_mode(gt_boxes, prompt_indices, mode):
    """
    Input : all GT boxes of the image, the indices used as visual prompts, mode
    Output: (evaluated_gt_boxes, prompt_gt_boxes)
            all_gt   -> (all boxes, empty)
            held_out -> (non-prompt boxes, prompt boxes)
    """
    gt_boxes = np.asarray(gt_boxes, dtype=np.float32).reshape(-1, 4)
    if mode == "all_gt":
        return gt_boxes, np.zeros((0, 4), dtype=np.float32)
    is_prompt = np.zeros(len(gt_boxes), dtype=bool)
    prompt_indices = np.asarray(prompt_indices, dtype=int)
    if len(prompt_indices):
        is_prompt[prompt_indices] = True
    return gt_boxes[~is_prompt], gt_boxes[is_prompt]


def evaluate_run_predictions(pred_boxes, pred_scores, eval_gt_boxes, prompt_gt_boxes,
                             eval_iou=EVAL_IOU_THRESHOLD, ignore_iou=PROMPT_IGNORE_IOU):
    """
    Evaluate ONE prediction set against ONE GT set.

    Output dict:
      status      (P,) int  - STATUS_TP / STATUS_FP / STATUS_IGNORED per prediction
      TP, FP, FN, n_ignored, n_eval_gt, n_pred
      precision, recall, F1
      IoU1 - mean IoU of the MATCHED prediction/GT pairs only
             ("when it finds a plant, how well is it localised?")
      IoU2 - sum of matched IoUs divided by the number of evaluated GT boxes
             ("localisation quality over ALL plants, missed ones count as 0")
      valid_for_macro - False when there is no GT left to evaluate (held_out runs
             in which every plant of the image was used as a prompt)
    """
    pred_boxes = np.asarray(pred_boxes, dtype=np.float32).reshape(-1, 4)
    pred_scores = np.asarray(pred_scores, dtype=np.float32).reshape(-1)
    eval_gt_boxes = np.asarray(eval_gt_boxes, dtype=np.float32).reshape(-1, 4)
    prompt_gt_boxes = np.asarray(prompt_gt_boxes, dtype=np.float32).reshape(-1, 4)

    n_pred, n_eval_gt = len(pred_boxes), len(eval_gt_boxes)
    status = np.full(n_pred, STATUS_FP, dtype=np.int8) # start eveything as FP then valid matches TP then leftover predictions on prompt as ignored and everything still FP

    # step 1 - corrected one-to-one matching against the evaluated GT
    match = match_one_to_one(pred_boxes, pred_scores, eval_gt_boxes, eval_iou)
    status[match["pred_match_gt"] >= 0] = STATUS_TP # Mark matched predictions as TP

    # step 2 - ignore the leftovers that sit on a PROMPT plant
    # Therefore prompt-ignore logic applies only to predictions that failed to match evaluated GT
    if n_pred and len(prompt_gt_boxes):
        leftover = np.where(status == STATUS_FP)[0] # Find predictions that are still FP (unmatched preds)
        if len(leftover):
            best_prompt_iou = compute_iou_matrix(pred_boxes[leftover], prompt_gt_boxes).max(axis=1)
            status[leftover[best_prompt_iou >= ignore_iou]] = STATUS_IGNORED

    tp = int((status == STATUS_TP).sum())
    fp = int((status == STATUS_FP).sum())          # step 3 - the rest are FP
    n_ignored = int((status == STATUS_IGNORED).sum())
    fn = int(n_eval_gt - tp)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / n_eval_gt if n_eval_gt > 0 else 0.0
    f1 = safe_f1(precision, recall)
    matched_ious = match["matched_ious"]
    iou1 = float(np.mean(matched_ious)) if matched_ious else 0.0
    iou2 = float(np.sum(matched_ious) / n_eval_gt) if n_eval_gt > 0 else 0.0

    return {
        "status": status, "pred_match_gt": match["pred_match_gt"],
        "TP": tp, "FP": fp, "FN": fn, "n_ignored": n_ignored,
        "n_eval_gt": n_eval_gt, "n_pred": n_pred,
        "precision": float(precision), "recall": float(recall), "F1": float(f1),
        "IoU1": iou1, "IoU2": iou2,
        "valid_for_macro": bool(n_eval_gt > 0),
    }


print("Evaluation core ready (modes:", EVALUATION_MODES, ")")

In [ ]:
# =============================================================================
# CELL 21 - AP50 AND AP50:95  (supervision.metrics.MeanAveragePrecision)
# =============================================================================
# WHY AP IS COMPUTED DIFFERENTLY FROM PRECISION / RECALL / F1
#
#   AP is an area under the precision-recall curve. That curve is produced by
#   walking through ALL detections ordered by confidence. Truncating the
#   detection list at an operating threshold would simply cut the tail off the
#   curve and report a smaller area - which says nothing about model quality.
#   Therefore AP always uses ALL saved predictions with score >= 0.30
#   (CONFIDENCE_THRESHOLD), after NMS.
#
#   Precision / recall / F1 / IoU1 / IoU2 describe ONE operating point: they
#   answer "if I deploy the detector with confidence >= c, what happens?".
#   Those use only the detections that survive the operating threshold.
#   In this experiment the operating threshold IS the inference threshold (0.30),
#   so both use exactly the same predictions - they still answer different
#   questions (ranking quality vs. deployed behaviour).
#
#   Both numbers can appear in the same row - they answer different questions.
# =============================================================================

# just a helper function that converts NumPy boxes into the format expected by Supervision
def make_detections(boxes, scores=None):
    boxes = np.asarray(boxes, dtype=np.float32).reshape(-1, 4)
    n = len(boxes)
    if scores is None:
        return sv.Detections(xyxy=boxes, class_id=np.zeros(n, dtype=int))
    return sv.Detections(xyxy=boxes,
                         confidence=np.asarray(scores, dtype=np.float32).reshape(-1),
                         class_id=np.zeros(n, dtype=int))


def compute_ap(pred_list, gt_list):
    """
    Input : two equally long lists of supervision Detections (predictions / GT).
            One entry = one evaluation episode (one run).
    Output: (AP50, AP50_95). NaN when there is no GT at all to evaluate.
    Passing several episodes at once gives the POOLED (dataset-level) AP, in which
    the detections of all episodes are ranked together in one PR curve.
    """
    if len(gt_list) == 0 or sum(len(g) for g in gt_list) == 0: # Check whether there is any GT
        return float("nan"), float("nan")
    try:
        # Create the Supervision metric (metric calculator, provides all pred and GTs, calculates the AP results)
        # Supervision exposes map50 for IoU=0.50 and map50_95 for IoUs 0.50:0.95
        result = MeanAveragePrecision().update(pred_list, gt_list).compute()
        ap50, ap5095 = float(result.map50), float(result.map50_95)
        # supervision returns -1.0 when a metric is undefined -> report NaN instead,
        # so it is excluded from means instead of dragging them down.
        return (ap50 if ap50 >= 0 else float("nan"),
                ap5095 if ap5095 >= 0 else float("nan"))
    except Exception as e:
        print("   (AP computation failed:", e, ")")
        return float("nan"), float("nan")


# Its job is not to calculate AP yet, Its job is to prepare: pred and GT for one run so they can later be given to compute_ap(...)
# nms_run: contains your detections after NMS
def ap_inputs_for_run(nms_run, eval_gt, prompt_gt):
    """
    Build the (prediction, GT) episode used for AP of ONE run.
    All post-NMS predictions with score >= CONFIDENCE_THRESHOLD are used;
    in held_out mode the predictions that were IGNORED (they belong to prompt
    plants) are removed first, exactly like in the operating-point evaluation.
    """
    ev = evaluate_run_predictions(nms_run["boxes"], nms_run["scores"], eval_gt, prompt_gt) # marks each pred as TP,FP,IGNORED, it cares only about the ignored predictions for AP calculation bcz in held_out mode , predictions corresponding to prompt rumex must be removed before AP
    keep = ev["status"] != STATUS_IGNORED  # Remove ignored predictions
    # Convert remaining predictions to Supervision format
    return (make_detections(nms_run["boxes"][keep], nms_run["scores"][keep]),
            make_detections(eval_gt))


print("AP helpers ready (AP always uses every prediction >= "
      f"{CONFIDENCE_THRESHOLD} after NMS).")

In [ ]:
# =============================================================================
# CELL 22 - OPERATING-POINT EVALUATION
# =============================================================================
# Choose one confidence threshold, remove predictions below it, then send the
# remaining predictions to CELL 20 for evaluation. This is the function that
# gives Precision, Recall, F1, IoU1, and IoU2 at one operating threshold.
# =============================================================================

def evaluate_at_operating_point(nms_run, eval_gt, prompt_gt, confidence_threshold):
    """
    Input : nms_run  - output of apply_nms_to_run (sorted by score, high -> low) => contains the detections after NMS            eval_gt / prompt_gt - from split_gt_for_mode
            confidence_threshold - the operating point being tested
    Output: the dict of evaluate_run_predictions for the thresholded predictions.
    """
    keep = nms_run["scores"] >= confidence_threshold # boolean mask where True entries are selected and the False entries are excluded
    return evaluate_run_predictions(nms_run["boxes"][keep], nms_run["scores"][keep],
                                    eval_gt, prompt_gt)


print("Operating configuration (fixed from the start, no sweep):")
print(f"  Confidence threshold = {CONFIDENCE_THRESHOLD:.2f}")
print(f"  NMS IoU threshold    = {NMS_IOU_THRESHOLD:.2f}")

In [ ]:
# =============================================================================
# CELL 23 - PLOTTING HELPERS
# =============================================================================
# Colours used everywhere:
#   yellow = GT Rumex boxes
#   cyan   = D1 exemplar prompt (most isolated of the three in embedding space)
#   lime   = D2 exemplar prompt
#   orange = D3 exemplar prompt
#   red    = predictions
# =============================================================================

COLOR_GT, COLOR_PRED = "yellow", "red"
ROLE_COLORS = {"D1": "cyan", "D2": "lime", "D3": "orange"}
ROLE_NAMES = {"D1": "diverse #1", "D2": "diverse #2", "D3": "diverse #3"}


def plot_confusion_matrix(tp, fp, fn, title, png_path):
    """2x2 detection confusion matrix; the background/background cell stays empty."""
    matrix = np.array([[tp, fn], [fp, np.nan]], dtype=float)
    fig, ax = plt.subplots(figsize=(5.2, 4.6))
    im = ax.imshow(np.nan_to_num(matrix, nan=0.0), cmap="Blues")
    ax.set_xticks([0, 1], ["Predicted\nRumex", "Predicted\nBackground"])
    ax.set_yticks([0, 1], ["Actual\nRumex", "Actual\nBackground"])
    labels = [[f"TP\n{tp}", f"FN\n{fn}"], [f"FP\n{fp}", "n/a\n(no true\nnegatives)"]]
    vmax = np.nanmax(matrix) if np.nanmax(matrix) > 0 else 1.0
    for i in range(2):
        for j in range(2):
            value = matrix[i, j]
            colour = "white" if (not np.isnan(value) and value > 0.5 * vmax) else "black"
            ax.text(j, i, labels[i][j], ha="center", va="center",
                    color=colour, fontsize=11)
    ax.set_title(title, fontsize=11)
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    fig.savefig(png_path, dpi=200)
    plt.show()
    plt.close(fig)


def _draw_boxes(ax, boxes, color, linewidth=1.5, linestyle="-", labels=None, fontsize=6):
    """Draw [x1,y1,x2,y2] boxes (display coordinates) on a matplotlib axis."""
    for k, (x1, y1, x2, y2) in enumerate(np.asarray(boxes).reshape(-1, 4)):
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                                       edgecolor=color, linewidth=linewidth,
                                       linestyle=linestyle))
        if labels is not None:
            ax.text(x1, y1 - 2, labels[k],
                    color="white" if color in ("red",) else "black",
                    fontsize=fontsize, va="bottom", ha="left",
                    bbox=dict(facecolor=color, edgecolor="none", pad=0.8, alpha=0.85))


def _display_copy(image, max_display_dim=PLOT_MAX_DISPLAY_DIM):
    """Downscaled numpy copy of the image + the factor to map full-res boxes onto it."""
    w, h = image.size
    s = min(1.0, max_display_dim / max(w, h))
    disp_w, disp_h = max(1, int(w * s)), max(1, int(h * s))
    display = np.asarray(image.resize((disp_w, disp_h), Image.BILINEAR))
    to_disp = np.array([disp_w / w, disp_h / h, disp_w / w, disp_h / h], dtype=np.float32)
    return display, to_disp


print("Plotting helpers ready.")

In [ ]:
# =============================================================================
# CELL 24 - PER-IMAGE METRICS  (one run = one image)
# =============================================================================
# The diversity choice is deterministic, so every image is evaluated exactly once and
# this single table replaces the run-level + image-level tables of the anchor
# experiments. Everything is evaluated at the fixed configuration.
#   AP50 / AP50_95 : all post-NMS predictions, confidence-ranked
#   P / R / F1 / IoU1 / IoU2 / TP / FP / FN : predictions >= CONFIDENCE_THRESHOLD
# =============================================================================

METRIC_COLUMNS = ["AP50", "AP50_95", "precision", "recall", "F1", "IoU1", "IoU2"]

image_rows = []
for run in runs:
    nms_run = apply_nms_to_run(run, NMS_IOU_THRESHOLD)
    gt = run["gt_boxes"]
    for mode in EVALUATION_MODES:
        eval_gt, prompt_gt = split_gt_for_mode(gt, run["prompt_indices"], mode)

        p_det, g_det = ap_inputs_for_run(nms_run, eval_gt, prompt_gt)
        ap50, ap5095 = compute_ap([p_det], [g_det])

        ev = evaluate_at_operating_point(nms_run, eval_gt, prompt_gt, CONFIDENCE_THRESHOLD)
        valid = ev["valid_for_macro"]
        nan = float("nan")

        image_rows.append({
            "experiment_name": EXPERIMENT_NAME,
            "image_ID": run["image_ID"],
            "Prompt_ID": run["Prompt_ID"],
            "Prompt_Type": run["Prompt_Type"],
            "evaluation_mode": mode,
            "confidence_threshold": CONFIDENCE_THRESHOLD,
            "nms_iou_threshold": NMS_IOU_THRESHOLD,
            "n_gt": len(gt),
            "n_prompt_gt": int(len(run["prompt_indices"])),
            "median_gt_area_px2": round(run["median_gt_area"]),
            "n_eval_gt": ev["n_eval_gt"],
            "n_tiles": run["n_tiles"],
            "n_predictions_pre_nms": nms_run["n_pre_nms"],
            "n_suppressed_cross_tile": nms_run["n_suppressed_cross_tile"],
            "n_suppressed_same_tile": nms_run["n_suppressed_same_tile"],
            "n_predictions": ev["n_pred"],
            "n_ignored_predictions": ev["n_ignored"],
            "AP50": ap50 if valid else nan,
            "AP50_95": ap5095 if valid else nan,
            "precision": ev["precision"] if valid else nan,
            "recall": ev["recall"] if valid else nan,
            "F1": ev["F1"] if valid else nan,
            "IoU1": ev["IoU1"] if valid else nan,
            "IoU2": ev["IoU2"] if valid else nan,
            "TP": ev["TP"], "FP": ev["FP"], "FN": ev["FN"],
            "valid_for_macro": valid,
        })

image_level_df = pd.DataFrame(image_rows).sort_values(
    ["evaluation_mode", "image_ID"]).reset_index(drop=True)
IMAGE_LEVEL_CSV = os.path.join(METRICS_DIR, "image_level_metrics.csv")
image_level_df.to_csv(IMAGE_LEVEL_CSV, index=False)

print(f"Per-image metrics: {len(image_level_df)} rows "
      f"({image_level_df['image_ID'].nunique()} images x {len(EVALUATION_MODES)} modes) "
      f"-> {IMAGE_LEVEL_CSV}")
for mode in EVALUATION_MODES:
    sub = image_level_df[image_level_df["evaluation_mode"] == mode]
    print(f"  {mode:9s}: {int(sub['valid_for_macro'].sum())} images valid for macro averaging, "
          f"AP50_mean={sub['AP50'].mean():.4f}, F1_mean={sub['F1'].mean():.4f}")

In [ ]:
# =============================================================================
# CELL 25 - EXPERIMENT-LEVEL SUMMARY
# =============================================================================
# Mean over images and std BETWEEN images of every per-image metric, so every
# UAV image contributes exactly the same weight regardless of how many GT boxes
# it contains.
# =============================================================================

summary_rows = []
for mode in EVALUATION_MODES:
    sub = image_level_df[image_level_df["evaluation_mode"] == mode]
    row = {
        "experiment_name": EXPERIMENT_NAME,
        "evaluation_mode": mode,
        "prompt_type": PROMPT_TYPE,
        "n_exemplars": K_EXEMPLARS,
        "embedding_model": DINOV2_MODEL_ID,
        "embedding_feature": EMBEDDING_FEATURE,
        "selection_rule": f"{DIVERSITY_OBJECTIVE}_{DIVERSITY_METRIC}",
        "crop_mode": CROP_MODE,
        "crop_context": CROP_CONTEXT,
        "use_tiling": USE_TILING,
        "tile_size": TILE_SIZE,
        "tile_overlap": OVERLAP,
        "confidence_threshold": CONFIDENCE_THRESHOLD,
        "nms_iou_threshold": NMS_IOU_THRESHOLD,
        "eval_iou_threshold": EVAL_IOU_THRESHOLD,
        "n_images": int(sub["image_ID"].nunique()),
        "n_images_valid_for_macro": int(sub["valid_for_macro"].sum()),
    }
    for col in METRIC_COLUMNS:
        row[f"{col}_mean"] = sub[col].mean()
        row[f"{col}_std"] = sub[col].std()      # spread between images
    summary_rows.append(row)

experiment_summary_df = pd.DataFrame(summary_rows)
EXPERIMENT_SUMMARY_CSV = os.path.join(METRICS_DIR, "experiment_summary.csv")
experiment_summary_df.to_csv(EXPERIMENT_SUMMARY_CSV, index=False)

print(f"Experiment summary -> {EXPERIMENT_SUMMARY_CSV}\n")
print(experiment_summary_df.to_string(index=False))

In [ ]:
# =============================================================================
# CELL 26 - POOLED DATASET AP50 / AP50:95
# =============================================================================
# This is NOT the mean of the per-image AP values. All images are handed to
# supervision at once, so every detection of the whole dataset is ranked in ONE
# precision-recall curve. Here one episode = one image.
# =============================================================================

dataset_rows = []
for mode in EVALUATION_MODES:
    pred_list, gt_list, images_used = [], [], set()
    for run in runs:
        nms_run = apply_nms_to_run(run, NMS_IOU_THRESHOLD)
        eval_gt, prompt_gt = split_gt_for_mode(run["gt_boxes"], run["prompt_indices"], mode)
        p_det, g_det = ap_inputs_for_run(nms_run, eval_gt, prompt_gt)
        pred_list.append(p_det)
        gt_list.append(g_det)
        images_used.add(run["image_ID"])
    ap50, ap5095 = compute_ap(pred_list, gt_list)   # one PR curve over ALL images
    dataset_rows.append({
        "experiment_name": EXPERIMENT_NAME,
        "evaluation_mode": mode,
        "prompt_type": PROMPT_TYPE,
        "n_images": len(images_used),
        "confidence_used_for_AP": CONFIDENCE_THRESHOLD,
        "nms_iou_threshold": NMS_IOU_THRESHOLD,
        "dataset_AP50": ap50,
        "dataset_AP50_95": ap5095,
    })
    del pred_list, gt_list
    gc.collect()

dataset_ap_df = pd.DataFrame(dataset_rows)
DATASET_AP_CSV = os.path.join(METRICS_DIR, "dataset_ap_metrics.csv")
dataset_ap_df.to_csv(DATASET_AP_CSV, index=False)

print(f"Dataset pooled AP -> {DATASET_AP_CSV}\n")
print(dataset_ap_df.to_string(index=False))
print("\nFor comparison, the MEAN of the per-image AP50 values (a different quantity):")
print(image_level_df.groupby("evaluation_mode")["AP50"].mean().to_string())

In [ ]:
# =============================================================================
# CELL 27 - DATASET-LEVEL CONFUSION MATRICES
# =============================================================================
# One class (Rumex) plus a background row/column:
#     Actual Rumex      -> Predicted Rumex      = TP
#     Actual Rumex      -> Predicted Background = FN  (missed plants)
#     Actual Background -> Predicted Rumex      = FP  (spurious detections)
#     Actual Background -> Predicted Background = not defined for detection
# Counts are pooled over every image at the fixed configuration. In held_out mode
# the D1/D2/D3 prompt plants and the detections that were ignored do not appear.
# =============================================================================

confusion_summary = []
for mode in EVALUATION_MODES:
    sub = image_level_df[image_level_df["evaluation_mode"] == mode]
    tp, fp, fn = int(sub["TP"].sum()), int(sub["FP"].sum()), int(sub["FN"].sum())
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = safe_f1(precision, recall)

    cm_df = pd.DataFrame(
        [[tp, fn], [fp, np.nan]],
        index=["actual_rumex", "actual_background"],
        columns=["predicted_rumex", "predicted_background"],
    )
    cm_df.to_csv(os.path.join(CONFUSION_MATRIX_DIR, f"confusion_matrix_{mode}.csv"))

    plot_confusion_matrix(
        tp, fp, fn,
        f"{EXPERIMENT_NAME} - {mode}\ndiversity prompts | conf={CONFIDENCE_THRESHOLD:.2f}, "
        f"NMS IoU={NMS_IOU_THRESHOLD:.2f}, eval IoU={EVAL_IOU_THRESHOLD:.2f}",
        os.path.join(CONFUSION_MATRIX_DIR, f"confusion_matrix_{mode}.png"))

    confusion_summary.append({
        "experiment_name": EXPERIMENT_NAME, "evaluation_mode": mode,
        "TP": tp, "FP": fp, "FN": fn,
        "precision_micro": precision, "recall_micro": recall, "F1_micro": f1,
        "confidence_threshold": CONFIDENCE_THRESHOLD, "nms_iou_threshold": NMS_IOU_THRESHOLD,
        "eval_iou_threshold": EVAL_IOU_THRESHOLD,
    })
    print(f"{mode:9s}: TP={tp}  FP={fp}  FN={fn}  "
          f"P={precision:.4f}  R={recall:.4f}  F1={f1:.4f}")

confusion_summary_df = pd.DataFrame(confusion_summary)
confusion_summary_df.to_csv(
    os.path.join(CONFUSION_MATRIX_DIR, "confusion_matrix_summary.csv"), index=False)
print("\nConfusion matrices saved to:", CONFUSION_MATRIX_DIR)

In [ ]:
# =============================================================================
# CELL 28 - RECALL PER SIZE GROUP (how do diversity prompts fare across plant sizes?)
# =============================================================================
# For every image the GT boxes are split at that image's MEDIAN area:
#   "area <= median" (the smaller half) and "area > median" (the larger half).
# Matching uses all_gt at the fixed operating point (same matcher as everywhere).
# The prompt plants themselves are reported separately, because SAM3 usually
# re-detects them and they would flatter the recall of their size group.
# NOTE: size is NOT the selection criterion in this experiment - this table shows
# whether three VISUALLY DIVERSE prompts happen to cover both size halves anyway,
# and is directly comparable with the same table of k_size_exemplars_no_tiling.
# =============================================================================

size_rows, prompt_rows = [], []
for run in runs:
    gt = run["gt_boxes"]
    nms_run = apply_nms_to_run(run, NMS_IOU_THRESHOLD)
    keep = nms_run["scores"] >= CONFIDENCE_THRESHOLD
    match = match_one_to_one(nms_run["boxes"][keep], nms_run["scores"][keep], gt,
                             EVAL_IOU_THRESHOLD)
    found = match["gt_match_pred"] >= 0

    areas = box_areas(gt)
    is_prompt = np.isin(np.arange(len(gt)), run["prompt_indices"])
    small_half = areas <= run["median_gt_area"]
    for name, grp in [("area <= median", small_half), ("area >  median", ~small_half)]:
        mask = grp & ~is_prompt                      # non-prompt GT boxes only
        size_rows.append({
            "image_ID": run["image_ID"], "size_group": name,
            "n_gt_non_prompt": int(mask.sum()), "found": int((found & mask).sum()),
            "recall": float((found & mask).sum() / mask.sum()) if mask.sum() else float("nan"),
        })
    for role, idx in zip(run["prompt_roles"], run["prompt_indices"]):
        prompt_rows.append({"image_ID": run["image_ID"], "role": role,
                            "gt_index": int(idx), "area_px2": float(areas[idx]),
                            "re_detected": bool(found[idx])})

size_group_df = pd.DataFrame(size_rows)
prompt_hits_df = pd.DataFrame(prompt_rows)
size_group_df.to_csv(os.path.join(METRICS_DIR, "size_group_recall.csv"), index=False)
prompt_hits_df.to_csv(os.path.join(METRICS_DIR, "prompt_plants_redetected.csv"), index=False)

pooled = (size_group_df.groupby("size_group")[["n_gt_non_prompt", "found"]].sum()
          .assign(recall_pooled=lambda d: d["found"] / d["n_gt_non_prompt"]))
per_image_mean = size_group_df.groupby("size_group")["recall"].mean().rename("recall_mean_over_images")

print("Recall of NON-PROMPT GT boxes per size group (all_gt matching):")
print(pooled.join(per_image_mean).to_string())
print("\nPrompt plants re-detected (share per role):")
print(prompt_hits_df.groupby("role")["re_detected"].agg(["count", "sum", "mean"]).to_string())
print("\nSaved:", os.path.join(METRICS_DIR, "size_group_recall.csv"), "and",
      os.path.join(METRICS_DIR, "prompt_plants_redetected.csv"))

In [ ]:
# =============================================================================
# CELL 29 - FINAL OUTPUT SUMMARY
# =============================================================================

print("=" * 78)
print(f"EXPERIMENT {EXPERIMENT_NAME} - FINAL SUMMARY")
print("=" * 78)
print(f"Prompts per image        : {K_EXEMPLARS} GT boxes selected by DINOv2 embedding "
      f"diversity (max-min cosine distance), no text, no negatives")
print(f"Embedding model          : {DINOV2_MODEL_ID} ({EMBEDDING_FEATURE} token, L2-normalised)")
print(f"Crops                    : {CROP_MODE} crop, context {CROP_CONTEXT:.2f}, full resolution")
print(f"Tiling                   : {USE_TILING}  (TILE_SIZE={TILE_SIZE}, OVERLAP={OVERLAP}, "
      f"{int(image_level_df['n_tiles'].mean())} tiles/image on average)")
print(f"Exemplar strip           : {EXEMPLAR_CROP_FOR_STRIP}, margin={STRIP_MARGIN}, "
      f"feather={FEATHER_WIDTH}, blur={BACKGROUND_BLUR_RADIUS}")
print(f"Confidence threshold     : {CONFIDENCE_THRESHOLD:.2f} (fixed; SAM3 run once per image)")
print(f"Mask threshold           : {MASK_THRESHOLD:.2f}")
print(f"NMS IoU threshold        : {NMS_IOU_THRESHOLD:.2f} (fixed)")
_cross = int(image_level_df.drop_duplicates("image_ID")["n_suppressed_cross_tile"].sum())
_same = int(image_level_df.drop_duplicates("image_ID")["n_suppressed_same_tile"].sum())
print(f"NMS suppressions         : {_cross} cross-tile + {_same} same-tile duplicates")
print(f"Evaluation IoU           : {EVAL_IOU_THRESHOLD:.2f}")
print(f"Images                   : {len(runs)}")
_min_d = np.array([r["min_pairwise_distance"] for r in runs], dtype=float)
_all_d = np.array([r["gt_mean_distance"] for r in runs], dtype=float)
print(f"Prompt diversity         : min pairwise distance mean={np.nanmean(_min_d):.3f} "
      f"(median={np.nanmedian(_min_d):.3f}), mean distance over ALL GT pairs="
      f"{np.nanmean(_all_d):.3f}")
print("-" * 78)
print("EXPERIMENT-LEVEL RESULTS (mean over images, std between images)")
show = ["evaluation_mode", "AP50_mean", "AP50_std", "AP50_95_mean", "precision_mean",
        "recall_mean", "F1_mean", "F1_std", "IoU1_mean", "IoU2_mean"]
print(experiment_summary_df[show].to_string(index=False))
print("-" * 78)
print("POOLED DATASET AP")
print(dataset_ap_df[["evaluation_mode", "dataset_AP50", "dataset_AP50_95"]].to_string(index=False))
print("-" * 78)
print("POOLED CONFUSION COUNTS")
print(confusion_summary_df[["evaluation_mode", "TP", "FP", "FN",
                            "precision_micro", "recall_micro", "F1_micro"]].to_string(index=False))
print("=" * 78)

print("\nFiles written under", RESULTS_ROOT)
for root, dirs, files in os.walk(RESULTS_ROOT):
    depth = root.replace(RESULTS_ROOT, "").count(os.sep)
    print("  " * depth + os.path.basename(root) + "/")
    if os.path.basename(root) == "raw_detections":
        npz_files = [f for f in files if f.endswith(".npz")]
        for f in sorted(files):
            if not f.endswith(".npz"):
                print("  " * (depth + 1) + f)
        print("  " * (depth + 1) + f"[{len(npz_files)} image NPZ files]")
    else:
        for f in sorted(files):
            print("  " * (depth + 1) + f)


In [ ]:
# =============================================================================
# CELL 30 - QUALITATIVE PLOT: BEST IMAGE, GT (left) vs PREDICTIONS (right)
# =============================================================================
# IMAGE SELECTION (per-image AP50, mode = PLOT_EVALUATION_MODE):
#   1. keep only images that have at least PLOT_MIN_GT_BOXES (7) GT boxes
#      and take the one with the highest AP50
#   2. if NO image has 7 GT boxes: keep the images with the HIGHEST number of GT
#      boxes and take the one among them with the highest AP50
#   ties are broken by F1, then by the number of GT boxes.
#
# Right panel = post-NMS predictions of that image with
# score >= CONFIDENCE_THRESHOLD, plus its D1 / D2 / D3 prompts (dashed).
# The image is downscaled for DISPLAY only (PLOT_MAX_DISPLAY_DIM).
# =============================================================================

def select_image_for_plot(mode=PLOT_EVALUATION_MODE, min_gt=PLOT_MIN_GT_BOXES):
    """Output: (per-image row, selection_rule_text) or (None, reason)."""
    img_df = image_level_df[(image_level_df["evaluation_mode"] == mode) &
                            image_level_df["AP50"].notna()].copy()
    if img_df.empty:
        return None, "no image with a valid AP50"

    candidates = img_df[img_df["n_gt"] >= min_gt]
    if len(candidates):
        rule = (f"highest AP50 among the {len(candidates)} images "
                f"with >= {min_gt} GT boxes")
    else:
        max_gt = int(img_df["n_gt"].max())
        candidates = img_df[img_df["n_gt"] == max_gt]
        rule = (f"no image has >= {min_gt} GT boxes -> highest AP50 among "
                f"the {len(candidates)} image(s) with the most GT boxes ({max_gt})")

    ranked = candidates.sort_values(["AP50", "F1", "n_gt"], ascending=False, na_position="last")
    print("Top candidates:")
    print(ranked[["image_ID", "n_gt", "Prompt_ID", "n_predictions", "AP50", "F1",
                  "precision", "recall"]].head(5).to_string(index=False))
    return ranked.iloc[0], rule


def plot_best_image(mode=PLOT_EVALUATION_MODE):
    row, rule = select_image_for_plot(mode)
    if row is None:
        print("Nothing to plot:", rule)
        return
    plot_image_id = row["image_ID"]
    run = runs_by_image[plot_image_id]

    nms_run = apply_nms_to_run(run, NMS_IOU_THRESHOLD)
    keep = nms_run["scores"] >= CONFIDENCE_THRESHOLD
    pred_boxes, pred_scores = nms_run["boxes"][keep], nms_run["scores"][keep]
    gt_boxes_plot = run["gt_boxes"]

    # ---- open the image, downscale for display only ---------------------------
    image_paths = {r[3]: r[1] for r in valid_images}
    if plot_image_id not in image_paths:
        print("Image file not found for", plot_image_id)
        return
    with Image.open(image_paths[plot_image_id]) as im:
        display, to_disp = _display_copy(im.convert("RGB"))

    # ---- figure: GT (left) | predictions (right) -------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(22, 8.5))
    for ax in axes:
        ax.imshow(display)
        ax.axis("off")

    gt_d = gt_boxes_plot * to_disp
    _draw_boxes(axes[0], gt_d, COLOR_GT, linewidth=1.5)
    axes[0].set_title(f"Ground truth: {len(gt_boxes_plot)} Rumex boxes", fontsize=12)

    score_labels = [f"{v:.2f}" for v in pred_scores] if PLOT_SHOW_SCORES else None
    _draw_boxes(axes[1], pred_boxes * to_disp, COLOR_PRED, linewidth=1.5, labels=score_labels)
    for role, idx in zip(run["prompt_roles"], run["prompt_indices"]):
        _draw_boxes(axes[1], gt_d[[idx]], ROLE_COLORS[role], linewidth=2.5, linestyle="--")
    axes[1].set_title(
        f"Predictions: {len(pred_boxes)} boxes | prompts {run['Prompt_ID']}\n"
        f"AP50={row['AP50']:.3f}  P={row['precision']:.3f}  R={row['recall']:.3f}  "
        f"F1={row['F1']:.3f}  TP={int(row['TP'])} FP={int(row['FP'])} FN={int(row['FN'])}",
        fontsize=12)

    legend_handles = [
        Line2D([0], [0], color=COLOR_GT, lw=2, label="ground truth"),
        Line2D([0], [0], color=COLOR_PRED, lw=2, label="prediction"),
    ] + [Line2D([0], [0], color=ROLE_COLORS[r], lw=2.5, linestyle="--",
                label=f"{r} prompt ({ROLE_NAMES[r]})") for r in run["prompt_roles"]]
    fig.legend(handles=legend_handles, loc="lower center", ncol=len(legend_handles),
               fontsize=11, frameon=False)
    fig.suptitle(
        f"{EXPERIMENT_NAME} | {plot_image_id} | mode={mode}\n"
        f"diversity prompts {run['Prompt_ID']} | min pairwise cosine distance "
        f"{run['min_pairwise_distance']:.3f} | {run['n_tiles']} tiles "
        f"(TILE_SIZE={TILE_SIZE}, OVERLAP={OVERLAP}) | "
        f"conf={CONFIDENCE_THRESHOLD:.2f}, NMS IoU={NMS_IOU_THRESHOLD:.2f}, "
        f"mask thr={MASK_THRESHOLD:.2f}\nselection: {rule}",
        fontsize=12)
    fig.tight_layout(rect=[0, 0.04, 1, 0.90])

    png_path = os.path.join(PLOTS_DIR, f"best_image_{safe_filename(plot_image_id)}_{mode}.png")
    fig.savefig(png_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print(f"\nSelected image : {plot_image_id} ({len(gt_boxes_plot)} GT boxes)")
    print(f"Selection rule : {rule}")
    print(f"Prompts        : {run['Prompt_ID']}")
    print(f"Figure saved   : {png_path}")


plot_best_image(PLOT_EVALUATION_MODE)